Run the MM output script but per LPS treatment

In [18]:
def list_csv_files_in_folder(folder_path):
    """
    Lists all CSV files in the specified folder and prints their names.
    
    Args:
        folder_path: Path to the folder to search
        
    Returns:
        List of CSV filenames found in the folder
    """
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found at {folder_path}")
        return []
    
    if not os.path.isdir(folder_path):
        print(f"ERROR: Path is not a directory: {folder_path}")
        return []
    
    # Get all CSV files in the folder
    csv_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.csv')]
    
    # Sort alphabetically
    csv_files.sort()
    
    print(f"\n{'='*60}")
    print(f"CSV Files in Folder")
    print(f"{'='*60}")
    print(f"Folder: {folder_path}")
    print(f"Found {len(csv_files)} CSV file(s):\n")
    
    for i, filename in enumerate(csv_files, 1):
        full_path = os.path.join(folder_path, filename)
        file_size = os.path.getsize(full_path)
        print(f"  {i:3d}. {filename} ({file_size:,} bytes)")
    
    print(f"{'='*60}\n")
    
    return csv_files


# ================================================================================
# USAGE EXAMPLE
# ================================================================================
FOLDER_TO_CHECK = '/Users/allisonpickle/Desktop/TimedLPS_1stBioRep'

csv_list = list_csv_files_in_folder(FOLDER_TO_CHECK)

# Optional: Print just the names as a list
if csv_list:
    print("\nFilenames only:")
    for name in csv_list:
        print(f"  '{name}'")


CSV Files in Folder
Folder: /Users/allisonpickle/Desktop/TimedLPS_1stBioRep
Found 35 CSV file(s):

    1. 24hr_Area (µm²).csv (15,022 bytes)
    2. 24hr_Aspect Ratio.csv (15,969 bytes)
    3. 24hr_Circularity.csv (16,220 bytes)
    4. 24hr_Convex Hull Area (µm²).csv (15,189 bytes)
    5. 24hr_Convex Hull Perimeter (µm).csv (15,960 bytes)
    6. 24hr_Perimeter (µm).csv (15,960 bytes)
    7. 24hr_Roughness.csv (15,940 bytes)
    8. 48hr_Area (µm²).csv (15,259 bytes)
    9. 48hr_Aspect Ratio.csv (15,907 bytes)
   10. 48hr_Circularity.csv (16,166 bytes)
   11. 48hr_Convex Hull Area (µm²).csv (15,045 bytes)
   12. 48hr_Convex Hull Perimeter (µm).csv (15,913 bytes)
   13. 48hr_Perimeter (µm).csv (15,909 bytes)
   14. 48hr_Roughness.csv (15,919 bytes)
   15. 6hr_Area (µm²).csv (13,578 bytes)
   16. 6hr_Aspect Ratio.csv (14,845 bytes)
   17. 6hr_Circularity.csv (15,086 bytes)
   18. 6hr_Convex Hull Area (µm²).csv (13,705 bytes)
   19. 6hr_Convex Hull Perimeter (µm).csv (14,846 bytes)
   20. 6

In [26]:
def combine_csv_files_from_folders(input_folders, output_folder, target_files=None):
    """
    Combines CSV files with matching names from multiple input folders into single files.
    
    Args:
        input_folders: List of 3 folder paths to search for CSV files
        output_folder: Path to save combined CSV files
        target_files: List of specific CSV filenames to combine (if None, combines all matching CSVs)
        
    Returns:
        Dictionary with results: {filename: {'success': bool, 'rows': int, 'sources': list}}
    """
    # Default target files if not specified
    if target_files is None:
        target_files = [
            '24hr_Area (µm²).csv',
            '24hr_Aspect Ratio.csv',
            '24hr_Circularity.csv',
            '24hr_Convex Hull Area (µm²).csv',
            '24hr_Convex Hull Perimeter (µm).csv',
            '24hr_Perimeter (µm).csv',
            '24hr_Roughness.csv',
            '48hr_Area (µm²).csv',
            '48hr_Aspect Ratio.csv',
            '48hr_Circularity.csv',
            '48hr_Convex Hull Area (µm²).csv',
            '48hr_Convex Hull Perimeter (µm).csv',
            '48hr_Perimeter (µm).csv',
            '48hr_Roughness.csv',
            '6hr_Area (µm²).csv',
            '6hr_Aspect Ratio.csv',
            '6hr_Circularity.csv',
            '6hr_Convex Hull Area (µm²).csv',
            '6hr_Convex Hull Perimeter (µm).csv',
            '6hr_Perimeter (µm).csv',
            '6hr_Roughness.csv',
            '72hr_Area (µm²).csv',
            '72hr_Aspect Ratio.csv',
            '72hr_Circularity.csv',
            '72hr_Convex Hull Area (µm²).csv',
            '72hr_Convex Hull Perimeter (µm).csv',
            '72hr_Perimeter (µm).csv',
            '72hr_Roughness.csv',
            '96hr_Area (µm²).csv',
            '96hr_Aspect Ratio.csv',
            '96hr_Circularity.csv',
            '96hr_Convex Hull Area (µm²).csv',
            '96hr_Convex Hull Perimeter (µm).csv',
            '96hr_Perimeter (µm).csv',
            '96hr_Roughness.csv'
        ]
    
    # Validate input folders
    for folder in input_folders:
        if not os.path.exists(folder):
            print(f"ERROR: Input folder not found: {folder}")
            return {}
        if not os.path.isdir(folder):
            print(f"ERROR: Path is not a directory: {folder}")
            return {}
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    print(f"\n{'='*70}")
    print("CSV File Combiner - Merging 3 Biological Replicates")
    print(f"{'='*70}")
    print(f"Input folders:")
    for i, folder in enumerate(input_folders, 1):
        print(f"  {i}. {folder}")
    print(f"\nOutput folder: {output_folder}")
    print(f"Target files: {len(target_files)}")
    print(f"{'='*70}\n")
    
    results = {}
    successful = 0
    failed = 0
    
    for target_file in target_files:
        print(f"\n{'─'*70}")
        print(f"Processing: {target_file}")
        print(f"{'─'*70}")
        
        dataframes = []
        found_in = []
        
        # Search for the file in each input folder
        for folder_idx, folder in enumerate(input_folders, 1):
            file_path = os.path.join(folder, target_file)
            
            if os.path.exists(file_path):
                try:
                    df = pd.read_csv(file_path)
                    dataframes.append(df)
                    found_in.append(f"Folder {folder_idx}")
                    print(f"  ✓ Found in Folder {folder_idx}: {len(df)} rows")
                except Exception as e:
                    print(f"  ✗ Error reading from Folder {folder_idx}: {e}")
            else:
                print(f"  ⚠ Not found in Folder {folder_idx}")
        
        # Combine dataframes if at least one was found
        if dataframes:
            try:
                combined_df = pd.concat(dataframes, ignore_index=True)
                output_path = os.path.join(output_folder, target_file)
                
                # Avoid overwriting by adding number if file exists
                output_path = _unique_output_path(output_path)
                
                combined_df.to_csv(output_path, index=False)
                
                print(f"\n  ✓ COMBINED: {len(combined_df)} total rows")
                print(f"  Sources: {', '.join(found_in)}")
                print(f"  Saved to: {os.path.basename(output_path)}")
                
                results[target_file] = {
                    'success': True,
                    'rows': len(combined_df),
                    'sources': found_in,
                    'output_path': output_path
                }
                successful += 1
                
            except Exception as e:
                print(f"\n  ✗ ERROR combining files: {e}")
                results[target_file] = {
                    'success': False,
                    'error': str(e)
                }
                failed += 1
        else:
            print(f"  ✗ File not found in any input folder")
            results[target_file] = {
                'success': False,
                'error': 'Not found in any folder'
            }
            failed += 1
    
    # Summary
    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"  Successfully combined: {successful}/{len(target_files)} files")
    print(f"  Failed: {failed}/{len(target_files)} files")
    print(f"  Output location: {output_folder}")
    print(f"{'='*70}\n")
    
    return results


# ================================================================================
# USAGE EXAMPLE
# ================================================================================
INPUT_FOLDERS = [
    '/Users/allisonpickle/Desktop/TimedLPS_1stBioRep',
    '/Users/allisonpickle/Desktop/TimedLPS_2ndBioRep',
    '/Users/allisonpickle/Desktop/TimedLPS_3rdBioRep'
]

OUTPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Compiled_Bio_Reps'

# Run the combination
results = combine_csv_files_from_folders(INPUT_FOLDERS, OUTPUT_FOLDER)

# Optional: Print detailed results
print("\nDetailed Results:")
for filename, info in results.items():
    if info['success']:
        print(f"✓ {filename}: {info['rows']} rows from {len(info['sources'])} folder(s)")
    else:
        print(f"✗ {filename}: {info.get('error', 'Unknown error')}")


CSV File Combiner - Merging 3 Biological Replicates
Input folders:
  1. /Users/allisonpickle/Desktop/TimedLPS_1stBioRep
  2. /Users/allisonpickle/Desktop/TimedLPS_2ndBioRep
  3. /Users/allisonpickle/Desktop/TimedLPS_3rdBioRep

Output folder: /Users/allisonpickle/Desktop/LPS_Compiled_Bio_Reps
Target files: 35


──────────────────────────────────────────────────────────────────────
Processing: 24hr_Area (µm²).csv
──────────────────────────────────────────────────────────────────────
  ✓ Found in Folder 1: 258 rows
  ✓ Found in Folder 2: 111 rows
  ✓ Found in Folder 3: 223 rows

  ✓ COMBINED: 592 total rows
  Sources: Folder 1, Folder 2, Folder 3
  Saved to: 24hr_Area (µm²).csv

──────────────────────────────────────────────────────────────────────
Processing: 24hr_Aspect Ratio.csv
──────────────────────────────────────────────────────────────────────
  ✓ Found in Folder 1: 258 rows
  ✓ Found in Folder 2: 111 rows
  ✓ Found in Folder 3: 223 rows

  ✓ COMBINED: 592 total rows
  Sources: F

In [33]:
# ================================================================================
# ORIGINAL FUNCTION - Compiles timepoints into single files by metric
# ================================================================================
def compile_csvs_by_metric(input_folder, output_folder=None):
    """
    Compiles CSV files across timepoints into single files grouped by metric.
    
    Takes CSV files named like "6hr_Circularity.csv", "24hr_Circularity.csv", etc.
    and combines all timepoints for each metric into a single CSV file.
    
    Modifies column B (Treatment) to contain first two underscore-separated
    elements from the Image filename (e.g., "6hr_100ng" from "6hr_100ng_LPS_01_2_C_0.tiff").
    
    Args:
        input_folder: Path to folder containing timepoint CSV files
        output_folder: Path to save compiled CSVs (if None, uses input_folder/Compiled_CSVs)
        
    Returns:
        Dictionary with compilation results: {metric_name: {'success': bool, 'output_path': str}}
    """
    import pandas as pd
    import re
    from collections import defaultdict
    
    if not os.path.exists(input_folder):
        print(f"ERROR: Input folder not found: {input_folder}")
        return {}
    
    # Set output folder
    if output_folder is None:
        output_folder = os.path.join(input_folder, 'Compiled_CSVs')
    os.makedirs(output_folder, exist_ok=True)
    
    # Define timepoint order
    timepoint_order = ['6hr', '24hr', '48hr', '72hr', '96hr']
    
    print(f"\n{'='*70}")
    print("CSV Compiler - Combining Timepoints by Metric")
    print(f"{'='*70}")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"{'='*70}\n")
    
    # Find all CSV files
    csv_files = [f for f in os.listdir(input_folder) 
                 if f.endswith('.csv') and os.path.isfile(os.path.join(input_folder, f))]
    
    if not csv_files:
        print("⚠ No CSV files found")
        return {}
    
    print(f"Found {len(csv_files)} CSV files\n")
    
    # Parse filenames and group by metric
    # Expected format: 6hr_Circularity.csv
    file_pattern = re.compile(r'^(\d+hr)_(.+)\.csv$')
    
    metric_groups = defaultdict(dict)  # {metric_name: {timepoint: file_path}}
    
    for filename in csv_files:
        match = file_pattern.match(filename)
        if match:
            timepoint = match.group(1)
            metric_name = match.group(2)
            
            file_path = os.path.join(input_folder, filename)
            metric_groups[metric_name][timepoint] = file_path
        else:
            print(f"⚠ Skipping file with unexpected format: {filename}")
    
    print(f"Grouped into {len(metric_groups)} unique metrics:\n")
    for metric in sorted(metric_groups.keys()):
        print(f"  • {metric}: {len(metric_groups[metric])} timepoints")
    
    # Compile CSVs for each metric
    results = {}
    
    for metric_name, timepoint_files in sorted(metric_groups.items()):
        print(f"\n{'─'*70}")
        print(f"Compiling: {metric_name}")
        print(f"{'─'*70}")
        
        dataframes = []
        
        # Read and collect data from each timepoint in order
        for timepoint in timepoint_order:
            if timepoint not in timepoint_files:
                print(f"  ⚠ {timepoint}: No file found")
                continue
            
            file_path = timepoint_files[timepoint]
            
            try:
                # Read CSV
                df = pd.read_csv(file_path)
                
                print(f"  DEBUG: Columns in {timepoint}: {list(df.columns)}")
                
                # Find the Image column (it might be named differently)
                image_col = None
                for col in df.columns:
                    if 'image' in col.lower() or 'filename' in col.lower() or 'file' in col.lower():
                        image_col = col
                        break
                
                if image_col:
                    print(f"  DEBUG: Found image column: '{image_col}'")
                    print(f"  DEBUG: Sample values: {df[image_col].head(3).tolist()}")
                    
                    # Extract first two underscore-separated elements from Image filename
                    # Example: "6hr_100ng_LPS_01_2_C_0.tiff" -> "6hr_100ng"
                    extracted_values = df[image_col].apply(
                        lambda x: '_'.join(str(x).split('_')[:2]) if pd.notna(x) else ''
                    )
                    
                    # Find column B (second column, index 1)
                    if len(df.columns) >= 2:
                        col_b_name = df.columns[1]
                        print(f"  DEBUG: Column B is '{col_b_name}', current values: {df[col_b_name].head(3).tolist()}")
                        
                        # Replace column B with extracted values
                        df[col_b_name] = extracted_values
                        
                        # Show a sample of the new values
                        sample_values = df[col_b_name].unique()[:3]
                        print(f"  ✓ {timepoint}: Modified '{col_b_name}' - New sample values: {list(sample_values)}")
                else:
                    print(f"  ⚠ {timepoint}: Could not find Image column")
                
                dataframes.append(df)
                print(f"  ✓ {timepoint}: Loaded {len(df)} rows from {os.path.basename(file_path)}")
                
            except Exception as e:
                print(f"  ✗ {timepoint}: Error reading file - {e}")
                import traceback
                traceback.print_exc()
                continue
        
        # Combine all dataframes
        if not dataframes:
            print(f"  ⚠ No data to compile for {metric_name}")
            results[metric_name] = {
                'success': False,
                'error': 'No valid dataframes found'
            }
            continue
        
        try:
            # Concatenate all timepoints
            combined_df = pd.concat(dataframes, ignore_index=True)
            
            # Save compiled CSV
            output_filename = f"Compiled_{metric_name}.csv"
            output_path = os.path.join(output_folder, output_filename)
            
            combined_df.to_csv(output_path, index=False)
            
            print(f"\n  ✓ COMPILED: {len(combined_df)} total rows from {len(dataframes)} timepoints")
            print(f"    Saved to: {output_filename}")
            
            # Show unique Treatment values in compiled file
            if len(combined_df.columns) >= 2:
                col_b_name = combined_df.columns[1]
                unique_treatments = combined_df[col_b_name].unique()
                print(f"    Unique '{col_b_name}' values: {len(unique_treatments)}")
                print(f"    Sample: {list(unique_treatments[:10])}")
            
            results[metric_name] = {
                'success': True,
                'output_path': output_path,
                'total_rows': len(combined_df),
                'n_timepoints': len(dataframes),
                'timepoints': [tp for tp in timepoint_order if tp in timepoint_files]
            }
            
        except Exception as e:
            print(f"\n  ✗ ERROR compiling {metric_name}: {e}")
            import traceback
            traceback.print_exc()
            results[metric_name] = {
                'success': False,
                'error': str(e)
            }
    
    # Summary
    successful = sum(1 for r in results.values() if r.get('success'))
    failed = len(results) - successful
    
    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"  Metrics processed: {len(metric_groups)}")
    print(f"  Successfully compiled: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output location: {output_folder}")
    print(f"{'='*70}\n")
    
    return results


# ================================================================================
# NEW FUNCTION - Splits compiled CSVs by second element of Treatment
# ================================================================================
def split_compiled_csvs_by_treatment_element(compiled_folder, output_folder=None):
    """
    Splits compiled CSV files by the second element of the Treatment column.
    
    Takes compiled CSVs with Treatment values like "6hr_5ug", "24hr_100ng", etc.
    and creates separate CSV files for each unique second element (5ug, 100ng, Tri, etc.).
    
    Example:
        Input: Compiled_Circularity.csv with treatments "6hr_5ug", "24hr_5ug", "48hr_100ng"
        Output: 
            - 5ug_Circularity.csv (contains 6hr_5ug, 24hr_5ug rows)
            - 100ng_Circularity.csv (contains 48hr_100ng rows)
    
    Args:
        compiled_folder: Path to folder containing Compiled_*.csv files
        output_folder: Path to save split CSVs (if None, uses compiled_folder/Split_By_Treatment)
        
    Returns:
        Dictionary with split results
    """
    import pandas as pd
    import re
    from collections import defaultdict
    
    if not os.path.exists(compiled_folder):
        print(f"ERROR: Compiled folder not found: {compiled_folder}")
        return {}
    
    # Set output folder
    if output_folder is None:
        output_folder = os.path.join(compiled_folder, 'Split_By_Treatment')
    os.makedirs(output_folder, exist_ok=True)
    
    print(f"\n{'='*70}")
    print("CSV Splitter - Splitting by Treatment Element (Second Part)")
    print(f"{'='*70}")
    print(f"Input folder: {compiled_folder}")
    print(f"Output folder: {output_folder}")
    print(f"{'='*70}\n")
    
    # Find all Compiled_*.csv files
    csv_files = [f for f in os.listdir(compiled_folder) 
                 if f.startswith('Compiled_') and f.endswith('.csv') 
                 and os.path.isfile(os.path.join(compiled_folder, f))]
    
    if not csv_files:
        print("⚠ No Compiled_*.csv files found")
        return {}
    
    print(f"Found {len(csv_files)} compiled CSV file(s)\n")
    
    results = {}
    
    for csv_file in csv_files:
        print(f"{'─'*70}")
        print(f"Processing: {csv_file}")
        print(f"{'─'*70}")
        
        file_path = os.path.join(compiled_folder, csv_file)
        
        # Extract metric name (remove "Compiled_" prefix and ".csv" suffix)
        metric_name = csv_file.replace('Compiled_', '').replace('.csv', '')
        
        try:
            # Read CSV
            df = pd.read_csv(file_path)
            
            # Check if Treatment column exists
            if len(df.columns) < 2:
                print(f"  ⚠ Skipping: Less than 2 columns\n")
                continue
            
            # Column B is the Treatment column (index 1)
            treatment_col = df.columns[1]
            
            print(f"  Treatment column: '{treatment_col}'")
            print(f"  Sample values: {df[treatment_col].head(3).tolist()}")
            
            # Extract second element from Treatment (e.g., "6hr_5ug" -> "5ug")
            df['TreatmentElement'] = df[treatment_col].apply(
                lambda x: str(x).split('_')[1] if pd.notna(x) and len(str(x).split('_')) > 1 else ''
            )
            
            # Get unique treatment elements
            treatment_elements = df['TreatmentElement'].unique()
            treatment_elements = [te for te in treatment_elements if te != '']
            
            if not treatment_elements:
                print(f"  ⚠ No valid treatment elements extracted\n")
                continue
            
            print(f"  Found {len(treatment_elements)} treatment element(s): {sorted(treatment_elements)}\n")
            
            # Split by treatment element and save
            for element in sorted(treatment_elements):
                df_element = df[df['TreatmentElement'] == element].copy()
                
                # Remove temporary column
                df_element = df_element.drop(columns=['TreatmentElement'])
                
                if len(df_element) == 0:
                    print(f"    ⚠ {element}: No rows (skipped)")
                    continue
                
                # Create output filename: element_metric.csv
                output_filename = f"{element}_{metric_name}.csv"
                output_path = os.path.join(output_folder, output_filename)
                
                # Save the split CSV
                df_element.to_csv(output_path, index=False)
                
                # Get timepoint distribution
                timepoints_in_element = df_element[treatment_col].apply(
                    lambda x: str(x).split('_')[0] if pd.notna(x) else ''
                ).unique()
                timepoints_in_element = sorted([t for t in timepoints_in_element if t])
                
                print(f"    ✓ {element}: {len(df_element)} rows -> {output_filename}")
                print(f"      Timepoints: {', '.join(timepoints_in_element)}")
            
            results[csv_file] = {
                'success': True,
                'metric_name': metric_name,
                'n_elements': len(treatment_elements),
                'elements': sorted(treatment_elements)
            }
            
            print()
            
        except Exception as e:
            print(f"  ✗ ERROR: {e}\n")
            import traceback
            traceback.print_exc()
            results[csv_file] = {
                'success': False,
                'error': str(e)
            }
    
    # Summary
    successful = sum(1 for r in results.values() if r.get('success'))
    failed = len(results) - successful
    
    print(f"{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"  Files processed: {len(csv_files)}")
    print(f"  Successfully split: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output location: {output_folder}")
    print(f"{'='*70}\n")
    
    return results


# ================================================================================
# USAGE EXAMPLES
# ================================================================================

# STEP 1: Compile timepoints into single files by metric
INPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Compiled_Bio_Reps'
OUTPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Compiled by Morphology Type'

print("\n" + "="*80)
print("STEP 1: COMPILING BY METRIC (COMBINING TIMEPOINTS)")
print("="*80)

csv_compilation_results = compile_csvs_by_metric(INPUT_FOLDER, OUTPUT_FOLDER)

print("\nDetailed CSV Compilation Results:")
for metric, info in csv_compilation_results.items():
    if info['success']:
        print(f"✓ {metric}: {info['total_rows']} rows from {info['n_timepoints']} timepoints ({', '.join(info['timepoints'])})")
    else:
        print(f"✗ {metric}: {info.get('error', 'Unknown error')}")


# STEP 2: Split compiled files by second element of treatment
COMPILED_FOLDER = OUTPUT_FOLDER
SPLIT_OUTPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Split_By_Treatment_Element'

print("\n" + "="*80)
print("STEP 2: SPLITTING BY TREATMENT ELEMENT (e.g., 5ug, 100ng, Tri)")
print("="*80)

split_results = split_compiled_csvs_by_treatment_element(COMPILED_FOLDER, SPLIT_OUTPUT_FOLDER)

print("\nDetailed Split Results:")
for file, info in split_results.items():
    if info['success']:
        print(f"✓ {info['metric_name']}: Split into {info['n_elements']} element(s) ({', '.join(info['elements'])})")
    else:
        print(f"✗ {file}: {info.get('error', 'Unknown error')}")


STEP 1: COMPILING BY METRIC (COMBINING TIMEPOINTS)

CSV Compiler - Combining Timepoints by Metric
Input folder: /Users/allisonpickle/Desktop/LPS_Compiled_Bio_Reps
Output folder: /Users/allisonpickle/Desktop/LPS_Compiled by Morphology Type

Found 35 CSV files

Grouped into 7 unique metrics:

  • Area (µm²): 5 timepoints
  • Aspect Ratio: 5 timepoints
  • Circularity: 5 timepoints
  • Convex Hull Area (µm²): 5 timepoints
  • Convex Hull Perimeter (µm): 5 timepoints
  • Perimeter (µm): 5 timepoints
  • Roughness: 5 timepoints

──────────────────────────────────────────────────────────────────────
Compiling: Area (µm²)
──────────────────────────────────────────────────────────────────────
  DEBUG: Columns in 6hr: ['Filename', 'Treatment', 'Biological_Rep', 'Well', 'Image', 'Results']
  DEBUG: Found image column: 'Filename'
  DEBUG: Sample values: ['6hr_100ng_LPS_01_1_C_0.tiff', '6hr_100ng_LPS_01_2_C_0.tiff', '6hr_100ng_LPS_01_3_C_0.tiff']
  DEBUG: Column B is 'Treatment', current values: 

In [34]:
# ================================================================================
# ORIGINAL FUNCTION - Compiles timepoints into single files by metric
# ================================================================================
def compile_csvs_by_metric(input_folder, output_folder=None):
    """
    Compiles CSV files across timepoints into single files grouped by metric.
    
    Takes CSV files named like "6hr_Circularity.csv", "24hr_Circularity.csv", etc.
    and combines all timepoints for each metric into a single CSV file.
    
    Modifies column B (Treatment) to contain first two underscore-separated
    elements from the Image filename (e.g., "6hr_100ng" from "6hr_100ng_LPS_01_2_C_0.tiff").
    
    Args:
        input_folder: Path to folder containing timepoint CSV files
        output_folder: Path to save compiled CSVs (if None, uses input_folder/Compiled_CSVs)
        
    Returns:
        Dictionary with compilation results: {metric_name: {'success': bool, 'output_path': str}}
    """
    import pandas as pd
    import re
    from collections import defaultdict
    
    if not os.path.exists(input_folder):
        print(f"ERROR: Input folder not found: {input_folder}")
        return {}
    
    # Set output folder
    if output_folder is None:
        output_folder = os.path.join(input_folder, 'Compiled_CSVs')
    os.makedirs(output_folder, exist_ok=True)
    
    # Define timepoint order
    timepoint_order = ['6hr', '24hr', '48hr', '72hr', '96hr']
    
    print(f"\n{'='*70}")
    print("CSV Compiler - Combining Timepoints by Metric")
    print(f"{'='*70}")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"{'='*70}\n")
    
    # Find all CSV files
    csv_files = [f for f in os.listdir(input_folder) 
                 if f.endswith('.csv') and os.path.isfile(os.path.join(input_folder, f))]
    
    if not csv_files:
        print("⚠ No CSV files found")
        return {}
    
    print(f"Found {len(csv_files)} CSV files\n")
    
    # Parse filenames and group by metric
    # Expected format: 6hr_Circularity.csv
    file_pattern = re.compile(r'^(\d+hr)_(.+)\.csv$')
    
    metric_groups = defaultdict(dict)  # {metric_name: {timepoint: file_path}}
    
    for filename in csv_files:
        match = file_pattern.match(filename)
        if match:
            timepoint = match.group(1)
            metric_name = match.group(2)
            
            file_path = os.path.join(input_folder, filename)
            metric_groups[metric_name][timepoint] = file_path
        else:
            print(f"⚠ Skipping file with unexpected format: {filename}")
    
    print(f"Grouped into {len(metric_groups)} unique metrics:\n")
    for metric in sorted(metric_groups.keys()):
        print(f"  • {metric}: {len(metric_groups[metric])} timepoints")
    
    # Compile CSVs for each metric
    results = {}
    
    for metric_name, timepoint_files in sorted(metric_groups.items()):
        print(f"\n{'─'*70}")
        print(f"Compiling: {metric_name}")
        print(f"{'─'*70}")
        
        dataframes = []
        
        # Read and collect data from each timepoint in order
        for timepoint in timepoint_order:
            if timepoint not in timepoint_files:
                print(f"  ⚠ {timepoint}: No file found")
                continue
            
            file_path = timepoint_files[timepoint]
            
            try:
                # Read CSV
                df = pd.read_csv(file_path)
                
                print(f"  DEBUG: Columns in {timepoint}: {list(df.columns)}")
                
                # Find the Image column (it might be named differently)
                image_col = None
                for col in df.columns:
                    if 'image' in col.lower() or 'filename' in col.lower() or 'file' in col.lower():
                        image_col = col
                        break
                
                if image_col:
                    print(f"  DEBUG: Found image column: '{image_col}'")
                    print(f"  DEBUG: Sample values: {df[image_col].head(3).tolist()}")
                    
                    # Extract first two underscore-separated elements from Image filename
                    # Example: "6hr_100ng_LPS_01_2_C_0.tiff" -> "6hr_100ng"
                    extracted_values = df[image_col].apply(
                        lambda x: '_'.join(str(x).split('_')[:2]) if pd.notna(x) else ''
                    )
                    
                    # Find column B (second column, index 1)
                    if len(df.columns) >= 2:
                        col_b_name = df.columns[1]
                        print(f"  DEBUG: Column B is '{col_b_name}', current values: {df[col_b_name].head(3).tolist()}")
                        
                        # Replace column B with extracted values
                        df[col_b_name] = extracted_values
                        
                        # Show a sample of the new values
                        sample_values = df[col_b_name].unique()[:3]
                        print(f"  ✓ {timepoint}: Modified '{col_b_name}' - New sample values: {list(sample_values)}")
                else:
                    print(f"  ⚠ {timepoint}: Could not find Image column")
                
                dataframes.append(df)
                print(f"  ✓ {timepoint}: Loaded {len(df)} rows from {os.path.basename(file_path)}")
                
            except Exception as e:
                print(f"  ✗ {timepoint}: Error reading file - {e}")
                import traceback
                traceback.print_exc()
                continue
        
        # Combine all dataframes
        if not dataframes:
            print(f"  ⚠ No data to compile for {metric_name}")
            results[metric_name] = {
                'success': False,
                'error': 'No valid dataframes found'
            }
            continue
        
        try:
            # Concatenate all timepoints
            combined_df = pd.concat(dataframes, ignore_index=True)
            
            # Save compiled CSV
            output_filename = f"Compiled_{metric_name}.csv"
            output_path = os.path.join(output_folder, output_filename)
            
            combined_df.to_csv(output_path, index=False)
            
            print(f"\n  ✓ COMPILED: {len(combined_df)} total rows from {len(dataframes)} timepoints")
            print(f"    Saved to: {output_filename}")
            
            # Show unique Treatment values in compiled file
            if len(combined_df.columns) >= 2:
                col_b_name = combined_df.columns[1]
                unique_treatments = combined_df[col_b_name].unique()
                print(f"    Unique '{col_b_name}' values: {len(unique_treatments)}")
                print(f"    Sample: {list(unique_treatments[:10])}")
            
            results[metric_name] = {
                'success': True,
                'output_path': output_path,
                'total_rows': len(combined_df),
                'n_timepoints': len(dataframes),
                'timepoints': [tp for tp in timepoint_order if tp in timepoint_files]
            }
            
        except Exception as e:
            print(f"\n  ✗ ERROR compiling {metric_name}: {e}")
            import traceback
            traceback.print_exc()
            results[metric_name] = {
                'success': False,
                'error': str(e)
            }
    
    # Summary
    successful = sum(1 for r in results.values() if r.get('success'))
    failed = len(results) - successful
    
    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"  Metrics processed: {len(metric_groups)}")
    print(f"  Successfully compiled: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output location: {output_folder}")
    print(f"{'='*70}\n")
    
    return results


# ================================================================================
# NEW FUNCTION - Compiles CSVs by treatment dose (second element of filename)
# ================================================================================
def compile_csvs_by_treatment_dose(input_folder, output_parent_folder=None):
    """
    Compiles CSV files by treatment dose, creating subfolders for each metric.
    
    Takes CSV files named like "6hr_Circularity.csv", "24hr_Circularity.csv" containing
    treatments like "6hr_100ng", "24hr_100ng", "6hr_5ug", etc. and creates:
    - A subfolder for each metric (Circularity, Area, etc.)
    - Within each subfolder, separate CSV files for each dose (100ng, 5ug, Tri, etc.)
    - Each dose file contains all timepoints for that dose
    
    Example output structure:
        Circularity/
            100ng_Circularity.csv  (6hr_100ng, 24hr_100ng, 48hr_100ng, etc.)
            5ug_Circularity.csv    (6hr_5ug, 24hr_5ug, etc.)
            Tri_Circularity.csv    (6hr_Tri, 24hr_Tri, etc.)
        Area/
            100ng_Area.csv
            5ug_Area.csv
            ...
    
    Args:
        input_folder: Path to folder containing timepoint CSV files (6hr_*.csv, 24hr_*.csv, etc.)
        output_parent_folder: Parent folder to create metric subfolders in
        
    Returns:
        Dictionary with compilation results
    """
    import pandas as pd
    import re
    from collections import defaultdict
    
    if not os.path.exists(input_folder):
        print(f"ERROR: Input folder not found: {input_folder}")
        return {}
    
    # Set output parent folder
    if output_parent_folder is None:
        output_parent_folder = os.path.join(input_folder, 'Compiled_By_Dose')
    os.makedirs(output_parent_folder, exist_ok=True)
    
    # Define timepoint order
    timepoint_order = ['6hr', '24hr', '48hr', '72hr', '96hr']
    
    print(f"\n{'='*70}")
    print("CSV Compiler - Organizing by Treatment Dose (Metric Subfolders)")
    print(f"{'='*70}")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_parent_folder}")
    print(f"{'='*70}\n")
    
    # Find all CSV files
    csv_files = [f for f in os.listdir(input_folder) 
                 if f.endswith('.csv') and os.path.isfile(os.path.join(input_folder, f))]
    
    if not csv_files:
        print("⚠ No CSV files found")
        return {}
    
    print(f"Found {len(csv_files)} CSV files\n")
    
    # Parse filenames and group by metric
    file_pattern = re.compile(r'^(\d+hr)_(.+)\.csv$')
    
    metric_groups = defaultdict(dict)  # {metric_name: {timepoint: file_path}}
    
    for filename in csv_files:
        match = file_pattern.match(filename)
        if match:
            timepoint = match.group(1)
            metric_name = match.group(2)
            
            file_path = os.path.join(input_folder, filename)
            metric_groups[metric_name][timepoint] = file_path
        else:
            print(f"⚠ Skipping file with unexpected format: {filename}")
    
    print(f"Grouped into {len(metric_groups)} unique metrics:\n")
    for metric in sorted(metric_groups.keys()):
        print(f"  • {metric}: {len(metric_groups[metric])} timepoints")
    
    # Process each metric
    results = {}
    
    for metric_name, timepoint_files in sorted(metric_groups.items()):
        print(f"\n{'─'*70}")
        print(f"Processing Metric: {metric_name}")
        print(f"{'─'*70}")
        
        # Create subfolder for this metric
        metric_output_folder = os.path.join(output_parent_folder, metric_name)
        os.makedirs(metric_output_folder, exist_ok=True)
        
        # Collect all data from all timepoints for this metric
        all_data = []
        
        for timepoint in timepoint_order:
            if timepoint not in timepoint_files:
                print(f"  ⚠ {timepoint}: No file found")
                continue
            
            file_path = timepoint_files[timepoint]
            
            try:
                df = pd.read_csv(file_path)
                
                # Find and modify Treatment column (column B)
                image_col = None
                for col in df.columns:
                    if 'image' in col.lower() or 'filename' in col.lower() or 'file' in col.lower():
                        image_col = col
                        break
                
                if image_col and len(df.columns) >= 2:
                    extracted_values = df[image_col].apply(
                        lambda x: '_'.join(str(x).split('_')[:2]) if pd.notna(x) else ''
                    )
                    col_b_name = df.columns[1]
                    df[col_b_name] = extracted_values
                
                all_data.append(df)
                print(f"  ✓ {timepoint}: Loaded {len(df)} rows")
                
            except Exception as e:
                print(f"  ✗ {timepoint}: Error - {e}")
                continue
        
        if not all_data:
            print(f"  ⚠ No data collected for {metric_name}\n")
            results[metric_name] = {'success': False, 'error': 'No data'}
            continue
        
        # Combine all timepoints
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Extract dose from Treatment column (second element: "6hr_100ng" -> "100ng")
        if len(combined_df.columns) >= 2:
            treatment_col = combined_df.columns[1]
            combined_df['Dose'] = combined_df[treatment_col].apply(
                lambda x: str(x).split('_')[1] if pd.notna(x) and len(str(x).split('_')) > 1 else ''
            )
            
            # Get unique doses
            doses = sorted([d for d in combined_df['Dose'].unique() if d != ''])
            
            if not doses:
                print(f"  ⚠ No valid doses extracted\n")
                results[metric_name] = {'success': False, 'error': 'No doses extracted'}
                continue
            
            print(f"\n  Found {len(doses)} dose(s): {', '.join(doses)}\n")
            
            # Split by dose and save to metric subfolder
            dose_results = {}
            for dose in doses:
                df_dose = combined_df[combined_df['Dose'] == dose].copy()
                df_dose = df_dose.drop(columns=['Dose'])
                
                if len(df_dose) == 0:
                    continue
                
                # Create output filename in metric subfolder
                output_filename = f"{dose}_{metric_name}.csv"
                output_path = os.path.join(metric_output_folder, output_filename)
                
                df_dose.to_csv(output_path, index=False)
                
                # Get timepoints in this dose
                timepoints_in_dose = df_dose[treatment_col].apply(
                    lambda x: str(x).split('_')[0] if pd.notna(x) else ''
                ).unique()
                timepoints_in_dose = sorted([t for t in timepoints_in_dose if t])
                
                print(f"    ✓ {dose}: {len(df_dose)} rows -> {metric_name}/{output_filename}")
                print(f"      Timepoints: {', '.join(timepoints_in_dose)}")
                
                dose_results[dose] = {
                    'rows': len(df_dose),
                    'timepoints': timepoints_in_dose,
                    'path': output_path
                }
            
            results[metric_name] = {
                'success': True,
                'metric_folder': metric_output_folder,
                'n_doses': len(dose_results),
                'doses': dose_results
            }
        
        print()
    
    # Summary
    successful = sum(1 for r in results.values() if r.get('success'))
    failed = len(results) - successful
    
    print(f"{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    print(f"  Metrics processed: {len(metric_groups)}")
    print(f"  Successfully compiled: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Output location: {output_parent_folder}")
    print(f"{'='*70}\n")
    
    return results


# ================================================================================
# USAGE EXAMPLES
# ================================================================================

# STEP 1: Compile timepoints into single files by metric (ORIGINAL)
INPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Compiled_Bio_Reps'
OUTPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Compiled by Morphology Type'

print("\n" + "="*80)
print("STEP 1: COMPILING BY METRIC (COMBINING ALL TIMEPOINTS)")
print("="*80)

csv_compilation_results = compile_csvs_by_metric(INPUT_FOLDER, OUTPUT_FOLDER)

print("\nDetailed CSV Compilation Results:")
for metric, info in csv_compilation_results.items():
    if info['success']:
        print(f"✓ {metric}: {info['total_rows']} rows from {info['n_timepoints']} timepoints ({', '.join(info['timepoints'])})")
    else:
        print(f"✗ {metric}: {info.get('error', 'Unknown error')}")


# STEP 2: Compile by treatment dose with metric subfolders (NEW)
DOSE_OUTPUT_FOLDER = '/Users/allisonpickle/Desktop/LPS_Compiled_By_Dose'

print("\n" + "="*80)
print("STEP 2: COMPILING BY DOSE (WITH METRIC SUBFOLDERS)")
print("="*80)

dose_compilation_results = compile_csvs_by_treatment_dose(INPUT_FOLDER, DOSE_OUTPUT_FOLDER)

print("\nDetailed Dose Compilation Results:")
for metric, info in dose_compilation_results.items():
    if info['success']:
        dose_list = ', '.join(info['doses'].keys())
        print(f"✓ {metric}: {info['n_doses']} dose(s) ({dose_list})")
        for dose, dose_info in info['doses'].items():
            print(f"    - {dose}: {dose_info['rows']} rows, {len(dose_info['timepoints'])} timepoints")
    else:
        print(f"✗ {metric}: {info.get('error', 'Unknown error')}")


STEP 1: COMPILING BY METRIC (COMBINING ALL TIMEPOINTS)

CSV Compiler - Combining Timepoints by Metric
Input folder: /Users/allisonpickle/Desktop/LPS_Compiled_Bio_Reps
Output folder: /Users/allisonpickle/Desktop/LPS_Compiled by Morphology Type

Found 35 CSV files

Grouped into 7 unique metrics:

  • Area (µm²): 5 timepoints
  • Aspect Ratio: 5 timepoints
  • Circularity: 5 timepoints
  • Convex Hull Area (µm²): 5 timepoints
  • Convex Hull Perimeter (µm): 5 timepoints
  • Perimeter (µm): 5 timepoints
  • Roughness: 5 timepoints

──────────────────────────────────────────────────────────────────────
Compiling: Area (µm²)
──────────────────────────────────────────────────────────────────────
  DEBUG: Columns in 6hr: ['Filename', 'Treatment', 'Biological_Rep', 'Well', 'Image', 'Results']
  DEBUG: Found image column: 'Filename'
  DEBUG: Sample values: ['6hr_100ng_LPS_01_1_C_0.tiff', '6hr_100ng_LPS_01_2_C_0.tiff', '6hr_100ng_LPS_01_3_C_0.tiff']
  DEBUG: Column B is 'Treatment', current valu

In [2]:
## USE THIS ONE

import os
import pandas as pd
import numpy as np

# --- helper function to avoid overwriting files ---
def _unique_output_path(path):
    """Generates a unique file path by appending a number if the file exists."""
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    counter = 1
    new_path = f"{base} ({counter}){ext}"
    while os.path.exists(new_path):
        counter += 1
        new_path = f"{base} ({counter}){ext}"
    return new_path


def _sanitize_filename_part(s: str) -> str:
    """Make a string safe-ish for filenames."""
    s = str(s).strip()
    for ch in ["/", "\\", ":", "*", "?", "\"", "<", ">", "|"]:
        s = s.replace(ch, "_")
    s = " ".join(s.split())
    return s


def _parse_treatment_well_image_from_filename(filename: str, treatment_tokens=(0, 1)):
    """
    From the filename in column A, parse underscore-separated tokens (before extension).

    EDITS IMPLEMENTED:
      - Treatment = specified tokens joined by "_" (default: parts[0] + "_" + parts[1])
        - Short filenames use available selected tokens, or parts[0] if none exist.
      - Well_raw  = 4th-to-last token (parts[-4])
      - Image_raw = 3rd-to-last token (parts[-3])

    Args:
      treatment_tokens: nonempty tuple of nonnegative, zero-based token indices
                        e.g., (0, 1) for first two, (1, 2, 3) for second through fourth

    Returns: (treatment, well_raw, image_raw)
      - well_raw / image_raw converted to int when possible; else kept as string.
    """
    if not treatment_tokens or any(not isinstance(i, int) or i < 0 for i in treatment_tokens):
        raise ValueError("treatment_tokens must contain nonnegative integer indices")

    filename = str(filename).strip()
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = [p for p in base.split("_") if p != ""]

    if not parts:
        return ("Unknown", "Unknown", "Unknown")

    # Extract treatment based on specified token positions
    selected_tokens = [parts[i] for i in treatment_tokens if i < len(parts)]
    treatment = "_".join(selected_tokens) if selected_tokens else parts[0]

    # Guard for short filenames
    if len(parts) < 4:
        return (treatment, "Unknown", "Unknown")

    well_tok = parts[-4]
    image_tok = parts[-3]

    try:
        well_raw = int(well_tok)
    except (ValueError, TypeError):
        well_raw = str(well_tok).strip()

    try:
        image_raw = int(image_tok)
    except (ValueError, TypeError):
        image_raw = str(image_tok).strip()

    return (treatment, well_raw, image_raw)


def _assign_well_numbers_change_based(
    filenames: pd.Series,
    sort_by_filename: bool = True,
    treatment_tokens=(0, 1),
) -> pd.Series:
    """
    Well numbering: simple sequential counter.

      - If sort_by_filename=True, compute wells in filename-sorted order.
      - Well starts at 1.
      - Well increments ONLY when the raw well token (4th-to-last filename element) 
        changes compared to the previous row.
      - The well NUMBER is independent of the actual well TOKEN value.
      - Returns a Series aligned to the original index of `filenames`.

    Example:
      Filename with well token "A1" -> Well 1
      Filename with well token "A1" -> Well 1 (same token)
      Filename with well token "B2" -> Well 2 (token changed)
      Filename with well token "B2" -> Well 2 (same token)
      
    This creates continuous well numbering across ALL samples in the sheet.
    """
    filenames = filenames.astype(str)

    if sort_by_filename:
        order = filenames.sort_values(kind="mergesort").index
    else:
        order = filenames.index

    prev_well_raw = object()  # Sentinel value that won't match anything
    current_well = 0
    wells = {}

    for i in order:
        fn = filenames.loc[i]
        # missing filename -> leave NA; it will be filtered later
        if fn.strip() == "" or fn.lower() == "nan":
            wells[i] = np.nan
            continue

        _, well_raw, _ = _parse_treatment_well_image_from_filename(fn, treatment_tokens)

        # Increment well counter whenever the well token changes
        if well_raw != prev_well_raw:
            current_well += 1
            prev_well_raw = well_raw

        wells[i] = current_well

    return pd.Series(wells).reindex(filenames.index)


def _assign_image_from_filename(filenames: pd.Series, treatment_tokens=(0, 1)) -> pd.Series:
    """
    REIMPLEMENTED image assignment:
      - Image is taken directly from the parsed 3rd-to-last token (int if possible, else string).
    """
    filenames = filenames.astype(str)
    images = []
    for fn in filenames:
        if fn.strip() == "" or fn.lower() == "nan":
            images.append(np.nan)
            continue
        _, _, image_raw = _parse_treatment_well_image_from_filename(fn, treatment_tokens)
        images.append(image_raw)
    return pd.Series(images, index=filenames.index)


def split_excel_sheet_multi_results(
    df: pd.DataFrame,
    sheet_name: str,
    input_excel_path: str,
    filename_col=0,                  # Column A by index
    results_cols=None,               # all other columns by default
    overwrite: bool = False,
    biological_rep=None,
    biological_rep_col=None,         # NEW: column index for bio rep (e.g., 8 for column I)
    output_dir=None,
    include_sheet_name_in_output: bool = True,
    sort_by_filename: bool = True,
    treatment_tokens=(0, 1),         # NEW: which tokens to use for treatment
):
    """
    For ONE sheet:
      - reads filenames from column A (by index)
      - for each metric column (all columns except column A by default), writes a separate CSV.

    Each output CSV contains columns:
      Filename, Treatment, Biological_Rep, Well, Image, Results

    EDITS IMPLEMENTED:
      - Treatment: specified tokens of filename (default: first 2 tokens)
      - Well: sequential counter (1, 2, 3...) that increments whenever the well token changes
              - Independent of bio rep
              - Independent of actual well token value
      - Image: parsed directly from filename token (-3)
      - Biological_Rep: 
        * If biological_rep_col is specified, read from that column (0-indexed)
        * Otherwise, use the biological_rep argument (sheet-level default)
      - treatment_tokens: tuple (e.g., (0, 1) or (1, 2, 3)) specifying which tokens form treatment
    """
    if df.shape[1] < 2:
        print(f"  Warning: Sheet has only {df.shape[1]} column(s). Need at least 2 (filename + results).")
        return {}

    if filename_col >= df.shape[1]:
        print(f"  Warning: filename_col index {filename_col} out of range for sheet with {df.shape[1]} columns.")
        return {}

    # Determine result columns
    if results_cols is None:
        results_cols = [c for c in range(df.shape[1]) if c != filename_col]
    else:
        results_cols = [c for c in results_cols if c < df.shape[1] and c != filename_col]

    # Exclude biological_rep_col from results if specified
    if biological_rep_col is not None and biological_rep_col in results_cols:
        results_cols.remove(biological_rep_col)

    if not results_cols:
        print("  Warning: No result columns found to export.")
        return {}

    # ---- choose output directory ----
    if output_dir is None:
        out_root = os.path.dirname(os.path.abspath(input_excel_path))
    else:
        out_root = os.path.abspath(output_dir)
        os.makedirs(out_root, exist_ok=True)

    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]
    safe_sheet_name = _sanitize_filename_part(sheet_name)

    # Filenames as a Series (keep original row mapping)
    filenames = df.iloc[:, filename_col]

    # NEW: Determine biological replicate source
    if biological_rep_col is not None:
        if biological_rep_col >= df.shape[1]:
            print(f"  Warning: biological_rep_col {biological_rep_col} out of range. Using default value.")
            bio_rep_series = pd.Series([biological_rep] * len(df), index=df.index)
        else:
            bio_rep_series = df.iloc[:, biological_rep_col]
            print(f"  Reading Biological_Rep from column {biological_rep_col} ('{df.columns[biological_rep_col]}')")
    else:
        bio_rep_series = pd.Series([biological_rep] * len(df), index=df.index)

    # Compute Treatment / Well / Image with the reimplemented rules
    treatment = []
    skipped_name_rows = 0

    for v in filenames:
        if pd.isna(v) or str(v).strip() == "":
            skipped_name_rows += 1
            treatment.append(np.nan)
        else:
            t, _, _ = _parse_treatment_well_image_from_filename(v, treatment_tokens)
            treatment.append(t)

    treatment = pd.Series(treatment, index=df.index)
    
    # Well numbering: sequential counter independent of bio rep
    well = _assign_well_numbers_change_based(filenames.fillna(""), sort_by_filename=sort_by_filename, treatment_tokens=treatment_tokens)
    image = _assign_image_from_filename(filenames.fillna(""), treatment_tokens=treatment_tokens)

    if skipped_name_rows > 0:
        print(f"  Skipped {skipped_name_rows} row(s) due to missing filename")

    per_metric_outputs = {}

    for col_idx in results_cols:
        metric_name = df.columns[col_idx]
        metric_name_safe = _sanitize_filename_part(metric_name)

        out_df = pd.DataFrame({
            "Filename": filenames.astype(str).str.strip(),
            "Treatment": treatment,
            "Biological_Rep": bio_rep_series,
            "Well": well,
            "Image": image,
            "Results": df.iloc[:, col_idx],
        })

        # Drop rows missing filename OR missing result
        out_df["Filename"] = out_df["Filename"].replace({"nan": np.nan, "": np.nan})
        before = len(out_df)
        out_df = out_df.dropna(subset=["Filename", "Results"]).reset_index(drop=True)
        skipped_results = before - len(out_df)

        if out_df.empty:
            print(f"  No valid rows to export for metric '{metric_name}' (column {col_idx}).")
            continue

        if include_sheet_name_in_output:
            out_path = os.path.join(out_root, f"{in_stem}_{safe_sheet_name}_{metric_name_safe}.csv")
        else:
            out_path = os.path.join(out_root, f"{in_stem}_{metric_name_safe}.csv")

        if not overwrite:
            out_path = _unique_output_path(out_path)

        out_df.to_csv(out_path, index=False)

        print(f"\n✓ Saved: {out_path}")
        print(f"  Sheet: {sheet_name}")
        print(f"  Metric: {metric_name} (col {col_idx})")
        print(f"  Treatment tokens: positions {treatment_tokens} (0-indexed)")
        print(f"  Biological Replicate: {'from column ' + str(biological_rep_col) if biological_rep_col is not None else biological_rep}")
        print(f"  Total rows: {len(out_df)}")
        if skipped_results > 0:
            print(f"  Skipped {skipped_results} row(s) due to missing filename/result values")

        per_metric_outputs[str(metric_name)] = (out_df, out_path)

    return per_metric_outputs


def split_excel_all_sheets_multi_results(
    input_excel_path,
    filename_col=0,
    results_cols=None,
    overwrite=False,
    biological_reps=None,
    biological_rep_col=None,         # NEW: column index to read bio rep from (e.g., 8 for column I)
    output_dir=None,
    include_sheet_name_in_output: bool = True,
    sort_by_filename: bool = True,
    treatment_tokens=(0, 1),         # NEW: which filename tokens to use for treatment
):
    """
    EDITS IMPLEMENTED: Bio rep assignment behavior (reimplemented)
      - biological_rep_col is int  -> read bio rep from that column (0-indexed) for ALL sheets
      - biological_reps is int     -> same bio rep for all sheets (if biological_rep_col is None)
      - biological_reps is dict    -> per-sheet bio rep, e.g. {"6hr": 1, "24hr": 1, ...}
      - biological_reps is None/other -> prompt per sheet (if biological_rep_col is None)
      - treatment_tokens: tuple specifying which filename tokens form treatment (e.g., (0, 1) or (1, 2, 3))
      
    Well numbering: Sequential counter (1, 2, 3...) that increments whenever well token changes
                    - Continues across ALL samples (all treatments, all bio reps)
      
    Priority: biological_rep_col > biological_reps
    """
    excel_file = pd.ExcelFile(input_excel_path)
    sheet_names = excel_file.sheet_names

    print(f"\n{'='*60}")
    print("Excel → Mixed Models CSV Converter")
    print(f"{'='*60}")
    print(f"Input file: {os.path.basename(input_excel_path)}")
    print(f"Found {len(sheet_names)} sheet(s):")
    for i, name in enumerate(sheet_names, 1):
        print(f"  {i}. {name}")
    print()
    print("Column mapping:")
    print(f"  Filenames: Column index {filename_col} (A)")
    print(f"  Treatment: Tokens {treatment_tokens} from filename (0-indexed)")
    print(f"  Results:   {('ALL other columns' if results_cols is None else list(results_cols))}")
    
    # NEW: Bio rep source reporting
    if biological_rep_col is not None:
        print(f"  Biological_Rep: From column index {biological_rep_col} (Excel column {chr(65 + biological_rep_col)})")
    elif isinstance(biological_reps, int):
        print(f"  Biological_Rep: Default value {biological_reps} for all sheets")
    elif isinstance(biological_reps, dict):
        print(f"  Biological_Rep: Per-sheet mapping provided")
    else:
        print(f"  Biological_Rep: Will prompt for each sheet")
    
    print(f"  Sort by filename: {sort_by_filename}")
    print(f"  Well numbering: Sequential counter across all samples (increments when well token changes)")
    if output_dir is not None:
        print(f"  Output:    {os.path.abspath(output_dir)}")
    print()

    def _bio_rep_for_sheet(sheet_name: str):
        """Only called if biological_rep_col is None"""
        if isinstance(biological_reps, dict):
            return biological_reps.get(sheet_name, None)
        if isinstance(biological_reps, int):
            return biological_reps
        while True:
            try:
                return int(input(f"Enter Biological Replicate number for sheet '{sheet_name}': "))
            except ValueError:
                print("Please enter a valid integer number.")

    all_results = {}

    for sheet_name in sheet_names:
        print(f"\n{'─'*60}")
        print(f"Processing sheet: {sheet_name}")
        print(f"{'─'*60}")

        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

        # Determine bio rep value (only used if biological_rep_col is None)
        bio_rep = _bio_rep_for_sheet(sheet_name) if biological_rep_col is None else None

        per_metric_outputs = split_excel_sheet_multi_results(
            df=df,
            sheet_name=sheet_name,
            input_excel_path=input_excel_path,
            filename_col=filename_col,
            results_cols=results_cols,
            overwrite=overwrite,
            biological_rep=bio_rep,
            biological_rep_col=biological_rep_col,
            output_dir=output_dir,
            include_sheet_name_in_output=include_sheet_name_in_output,
            sort_by_filename=sort_by_filename,
            treatment_tokens=treatment_tokens,
        )

        if per_metric_outputs:
            all_results[sheet_name] = per_metric_outputs

    print(f"\n{'='*60}")
    print(f"SUMMARY: Processed {len(all_results)} sheet(s)")
    print(f"{'='*60}")
    for sheet_name, metrics in all_results.items():
        print(f"  {sheet_name}:")
        for metric_name, (df_out, path) in metrics.items():
            print(f"    - {metric_name}: {len(df_out)} rows → {os.path.basename(path)}")
    print(f"{'='*60}\n")

    return all_results


# ================================================================================
# USAGE EXAMPLES
# ================================================================================
EXCEL_FILE_PATH = '/Users/allisonpickle/Desktop/Old Compiled w3/Compiled 5ugLPS 10uM w3/Isolated microglia/Perimeter_Area_ConvexHull_Circularity_AspectRoughness_Results.xlsx'
OUTPUT_DIR ='/Users/allisonpickle/Desktop/Old Compiled w3/Compiled 5ugLPS 10uM w3/Isolated microglia'

if not os.path.exists(EXCEL_FILE_PATH):
    print(f"ERROR: File not found at {EXCEL_FILE_PATH}")
else:
    # ============================================================
    # OPTION 1: Treatment from FIRST TWO tokens (default)
    # Example filename: Tri_Ctrl_001_A1_1_C_0.tiff
    # Treatment will be: "Tri_Ctrl"
    # ============================================================
    # results = split_excel_all_sheets_multi_results(
    #     EXCEL_FILE_PATH,
    #     filename_col=0,
    #     results_cols=None,
    #     overwrite=False,
    #     biological_rep_col=None,
    #     output_dir=OUTPUT_DIR,
    #     include_sheet_name_in_output=True,
    #     sort_by_filename=True,
    #     treatment_tokens=(0, 1),     # <-- First two tokens (0 and 1)
    # )

    
    # # ============================================================
    # # OPTION 2: Treatment from SECOND and THIRD tokens
    # # Example filename: 48hr_Tri_Ctrl_001_A1_1_C_0.tiff
    # # Treatment will be: "Tri_Ctrl"
    # # ============================================================
    # results = split_excel_all_sheets_multi_results(
    #     EXCEL_FILE_PATH,
    #     filename_col=0,
    #     results_cols=None,
    #     overwrite=False,
    #     biological_reps=1,
    #     biological_rep_col=None,
    #     output_dir=OUTPUT_DIR,
    #     include_sheet_name_in_output=True,
    #     sort_by_filename=True,
    #     treatment_tokens=(0, 1),     # <-- Second and third tokens (1 and 2)
    # )
    
    # # ============================================================
    # # OPTION 3: Read biological_rep from column B (index 1)
    # # ============================================================
    results = split_excel_all_sheets_multi_results(
        EXCEL_FILE_PATH,
        filename_col=0,
        results_cols=None,
        overwrite=False,
        biological_reps=None,
        biological_rep_col=1,        # <-- Column B (0-indexed)
        output_dir=OUTPUT_DIR,
        include_sheet_name_in_output=False,
        sort_by_filename=True,
        treatment_tokens=(2, 3),     # <-- Can also use (1, 2) if needed
    )

    # ============================================================
    # OPTION 4: Treatment from THREE tokens; bio rep from column B
    # Example filename: 3_5mM_PD_0LPS_11_5_C_0.tiff
    # Treatment will be: "5mM_PD_0LPS"
    # Comment out Option 3 above before enabling this call.
    # ============================================================
    # results = split_excel_all_sheets_multi_results(
    #     EXCEL_FILE_PATH,
    #     filename_col=0,
    #     results_cols=None,
    #     overwrite=False,
    #     biological_reps=None,
    #     biological_rep_col=1,
    #     output_dir=OUTPUT_DIR,
    #     include_sheet_name_in_output=True,
    #     sort_by_filename=True,
    #     treatment_tokens=(1, 2, 3),
    # )


Excel → Mixed Models CSV Converter
Input file: Perimeter_Area_ConvexHull_Circularity_AspectRoughness_Results.xlsx
Found 1 sheet(s):
  1. Sheet1

Column mapping:
  Filenames: Column index 0 (A)
  Treatment: Tokens (2, 3) from filename (0-indexed)
  Results:   ALL other columns
  Biological_Rep: From column index 1 (Excel column B)
  Sort by filename: True
  Well numbering: Sequential counter across all samples (increments when well token changes)
  Output:    /Users/allisonpickle/Desktop/Old Compiled w3/Compiled 5ugLPS 10uM w3/Isolated microglia


────────────────────────────────────────────────────────────
Processing sheet: Sheet1
────────────────────────────────────────────────────────────
  Reading Biological_Rep from column 1 ('Bio Rep')

✓ Saved: /Users/allisonpickle/Desktop/Old Compiled w3/Compiled 5ugLPS 10uM w3/Isolated microglia/Perimeter_Area_ConvexHull_Circularity_AspectRoughness_Results_Base Name.csv
  Sheet: Sheet1
  Metric: Base Name (col 2)
  Treatment tokens: positions 

For Modular Tri-Culture Below:

In [ ]:
# ============================================================
# Excel to Mixed Models CSV Converter
# - Reads an Excel file with multiple sheets
# - Uses a "Treatment" sheet:
#     Col A = Filename
#     Col B = Treatment
#     Col C = Biological_Rep
# - Applies Treatment + Biological_Rep to all other sheets
# - Writes one formatted CSV per sheet (without overwriting existing files)
#
# EDIT (2026-04-14):
#   Revert the "expand rows for duplicate filename keys" behavior.
#   Instead:
#     - build a SINGLE mapping: filename_key -> (treatment, biological_rep)
#     - if duplicates exist, warn and keep the FIRST occurrence (do not inflate row counts)
#     - simply copy Treatment/BioRep onto each data row by filename match
# ============================================================
from __future__ import annotations
import os
import pandas as pd
import numpy as np
from datetime import datetime


def _unique_output_path(path):
    """Generates a unique file path by appending a number if the file exists."""
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    counter = 1
    new_path = f"{base} ({counter}){ext}"
    while os.path.exists(new_path):
        counter += 1
        new_path = f"{base} ({counter}){ext}"
    return new_path


def _well_sort_key(x: str):
    try:
        return (0, int(x))
    except ValueError:
        return (1, x)


def _normalize_filename_key(name: str) -> str:
    name = str(name).strip()
    name = os.path.basename(name)
    name = os.path.splitext(name)[0]
    return name


def load_treatment_map_from_sheet(
    input_excel_path: str,
    *,
    treatment_sheet_name: str = "Treatment",
    filename_col: int = 0,
    treatment_col: int = 1,
    biorep_col: int = 2,
    on_duplicate: str = "first",  # "first" | "last" | "error"
) -> dict[str, tuple[str, int | None]]:
    """
    SINGLE mapping:
      normalized_filename_key -> (treatment, biological_rep)

    This does NOT expand rows. If duplicates exist in the Treatment sheet for the same
    normalized key, we keep the first by default (and warn).
    """
    df_map = pd.read_excel(input_excel_path, sheet_name=treatment_sheet_name)

    if df_map.shape[1] <= max(filename_col, treatment_col, biorep_col):
        raise ValueError(
            f"Treatment sheet '{treatment_sheet_name}' does not have enough columns. "
            f"Need at least {max(filename_col, treatment_col, biorep_col) + 1} column(s)."
        )

    mapping: dict[str, tuple[str, int | None]] = {}
    skipped = 0
    duplicates = 0

    for i in range(len(df_map)):
        fname = df_map.iloc[i, filename_col]
        trt = df_map.iloc[i, treatment_col]
        bio = df_map.iloc[i, biorep_col]

        if pd.isna(fname) or pd.isna(trt):
            skipped += 1
            continue

        key = _normalize_filename_key(fname)
        treatment = str(trt).strip()

        bio_rep: int | None = None
        if not pd.isna(bio):
            try:
                bio_rep = int(bio)
            except (ValueError, TypeError):
                bio_rep = None

        if key in mapping:
            duplicates += 1
            if on_duplicate == "first":
                continue
            if on_duplicate == "last":
                mapping[key] = (treatment, bio_rep)
                continue
            if on_duplicate == "error":
                raise ValueError(f"Duplicate filename key in Treatment sheet: '{key}'")
        else:
            mapping[key] = (treatment, bio_rep)

    if not mapping:
        raise ValueError(f"No usable filename->(treatment,biorep) rows found in sheet '{treatment_sheet_name}'.")

    if skipped:
        print(f"⚠️  Treatment map: skipped {skipped} row(s) with missing filename or treatment.")
    if duplicates:
        print(
            f"⚠️  Treatment map: {duplicates} duplicate filename key(s) found in Treatment sheet. "
            f"Kept '{on_duplicate}' occurrence to avoid inflating output rows."
        )

    print(f"✓ Loaded treatment map from sheet '{treatment_sheet_name}': {len(mapping)} unique filename key(s)")
    return mapping


def split_excel_sheet(
    df,
    sheet_name,
    input_excel_path,
    filename_col=0,
    value_col=1,
    overwrite=False,
    biological_rep=None,  # fallback only
    start_well=1,
    *,
    treatment_map: dict[str, tuple[str, int | None]] | None = None,
):
    if df.shape[1] < 2:
        print(f"  ⚠️  Warning: Sheet has only {df.shape[1]} column(s). Need at least 2 (filename + results).")
        return pd.DataFrame(), None, start_well

    filename_col_name = df.columns[filename_col] if filename_col < len(df.columns) else f"Column {filename_col}"
    value_col_name = df.columns[value_col] if value_col < len(df.columns) else f"Column {value_col}"

    print(f"  Column {filename_col} (Filename): '{filename_col_name}'")
    print(f"  Column {value_col} (Results):   '{value_col_name}'")

    out_rows = []
    skipped_rows = 0
    missing_trt_rows = 0
    missing_bio_rows = 0
    parse_warn_rows = 0

    for idx in range(len(df)):
        name = df.iloc[idx, filename_col]
        if pd.isna(name):
            skipped_rows += 1
            continue
        name = str(name).strip()

        key = _normalize_filename_key(name)
        parts = key.split("_")

        if len(parts) >= 5:
            well_token = parts[-5]
            image_token = parts[-4]
        else:
            parse_warn_rows += 1
            well_token = None
            image_token = None

        try:
            well_num = int(well_token) if well_token is not None else None
        except (ValueError, TypeError):
            well_num = well_token

        try:
            image_num = int(image_token) if image_token is not None else None
        except (ValueError, TypeError):
            image_num = image_token

        result_val = df.iloc[idx, value_col]
        if pd.isna(result_val):
            print(f"  ⚠️  Row {idx+1}: Missing result value for {name}")
            skipped_rows += 1
            continue

        treatment = "Unknown"
        bio_rep_row = biological_rep

        if treatment_map is not None:
            hit = treatment_map.get(key, None)
            if hit is None:
                missing_trt_rows += 1
            else:
                treatment, bio_from_map = hit
                if bio_from_map is not None:
                    bio_rep_row = bio_from_map

        if bio_rep_row is None:
            missing_bio_rows += 1

        out_rows.append(
            {
                "Filename": name,
                "Treatment": treatment,
                "Biological_Rep": bio_rep_row,
                "Well": well_num,
                "Image": image_num,
                "Results": result_val,
                "Original_Order": idx,
            }
        )

    if skipped_rows > 0:
        print(f"  ⚠️  Skipped {skipped_rows} row(s) due to missing filename or result value")
    if missing_trt_rows > 0 and treatment_map is not None:
        print(f"  ⚠️  {missing_trt_rows} row(s) had no filename match in Treatment sheet; Treatment set to 'Unknown'")
    if missing_bio_rows > 0:
        print(f"  ⚠️  {missing_bio_rows} row(s) have no Biological_Rep (neither from map nor fallback)")
    if parse_warn_rows > 0:
        print(f"  ⚠️  {parse_warn_rows} row(s) had too few '_' parts to parse Well/Image by position; set to None")

    out_df = pd.DataFrame(out_rows)
    if len(out_df) == 0:
        print("  ⚠️  No valid rows to export")
        return pd.DataFrame(), None, start_well

    out_df = out_df.sort_values(["Original_Order"], na_position="last").reset_index(drop=True)
    out_df = out_df[["Filename", "Treatment", "Biological_Rep", "Well", "Image", "Results"]]

    in_dir = os.path.dirname(os.path.abspath(input_excel_path))
    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]
    safe_sheet_name = str(sheet_name).replace("/", "_").replace("\\", "_")
    out_path = os.path.join(in_dir, f"{in_stem}_{safe_sheet_name}_RMixedFormated.csv")
    if not overwrite:
        out_path = _unique_output_path(out_path)

    out_df.to_csv(out_path, index=False)
    print(f"\n✓ Saved: {out_path}")
    print(f"  Sheet: {sheet_name}")
    print(f"  Total rows: {len(out_df)}")
    return out_df, out_path, start_well


def split_excel_all_sheets(
    input_excel_path,
    filename_col=0,
    value_col=1,
    overwrite=False,
    biological_reps=None,
    auto_number_bio_reps=True,
    *,
    treatment_sheet_name: str = "Treatment",
):
    excel_file = pd.ExcelFile(input_excel_path)
    sheet_names = excel_file.sheet_names

    treatment_map = None
    if treatment_sheet_name in sheet_names:
        treatment_map = load_treatment_map_from_sheet(
            input_excel_path,
            treatment_sheet_name=treatment_sheet_name,
            filename_col=filename_col,
            treatment_col=1,
            biorep_col=2,
            on_duplicate="first",
        )
    else:
        print(f"⚠️  No '{treatment_sheet_name}' sheet found; Treatment will be 'Unknown'.")

    data_sheet_names = [s for s in sheet_names if s != treatment_sheet_name]

    print(f"\n{'='*60}")
    print("Excel to Mixed Models CSV Converter")
    print(f"{'='*60}")
    print(f"Input file: {os.path.basename(input_excel_path)}")
    print(f"Found {len(sheet_names)} sheet(s): {sheet_names}")
    print(f"Processing {len(data_sheet_names)} data sheet(s): {data_sheet_names}\n")

    results = {}

    for sheet_name in data_sheet_names:
        print(f"\n{'─'*60}")
        print(f"Processing sheet: {sheet_name}")
        print(f"{'─'*60}")

        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

        if isinstance(biological_reps, dict):
            bio_rep = biological_reps.get(sheet_name, None)
        elif isinstance(biological_reps, int):
            bio_rep = biological_reps
        else:
            bio_rep = None

        out_df, out_path, _ = split_excel_sheet(
            df=df,
            sheet_name=sheet_name,
            input_excel_path=input_excel_path,
            filename_col=filename_col,
            value_col=value_col,
            overwrite=overwrite,
            biological_rep=bio_rep,
            treatment_map=treatment_map,
        )

        if out_path is not None:
            results[sheet_name] = (out_df, out_path)

    return results


# ================================================================================
# CALL THE SOURCE EXCEL FILE
# ================================================================================
EXCEL_FILE_PATH = '/Users/allisonpickle/Desktop/260701_Excitotoxicity/All/Isolated microglia/Results.xlsx'
if not os.path.exists(EXCEL_FILE_PATH):
    print(f"ERROR: File not found at {EXCEL_FILE_PATH}")
    print("Please check the file path.")
else:
    results = split_excel_all_sheets(
        EXCEL_FILE_PATH,
        filename_col=0,
        value_col=1,
        biological_reps=None,
        treatment_sheet_name="Treatment",
    )

⚠️  No 'Treatment' sheet found; Treatment will be 'Unknown'.

Excel to Mixed Models CSV Converter
Input file: Perimeter_Area_ConvexHull_Circularity_AspectRoughness_Results.xlsx
Found 7 sheet(s): ['OG', '6hr', '24hr', '48hr', '72hr', '96hr', 'Old LPS Comp']
Processing 7 data sheet(s): ['OG', '6hr', '24hr', '48hr', '72hr', '96hr', 'Old LPS Comp']


────────────────────────────────────────────────────────────
Processing sheet: OG
────────────────────────────────────────────────────────────
  Column 0 (Filename): 'File'
  Column 1 (Results):   'Base Name'
  ⚠️  1115 row(s) have no Biological_Rep (neither from map nor fallback)

✓ Saved: /Users/allisonpickle/Desktop/260630_TimedLPSOnly/All/Isolated microglia/Perimeter_Area_ConvexHull_Circularity_AspectRoughness_Results_OG_RMixedFormated.csv
  Sheet: OG
  Total rows: 1115

────────────────────────────────────────────────────────────
Processing sheet: 6hr
────────────────────────────────────────────────────────────
  Column 0 (Filename): 'Fil

Code for No AB

In [5]:
# ============================================================
# Excel to Mixed Models CSV Converter
# - Reads an Excel file with multiple sheets
# - Uses a "Treatment" sheet:
#     Col A = Filename
#     Col B = Treatment
#     Col C = Biological_Rep
# - Applies Treatment + Biological_Rep to all other sheets
# - Writes one formatted CSV per sheet (without overwriting existing files)
#
# Parsing rule (UPDATED, per request):
#   For filenames like:
#     TgDIV18_032124_WT11_Abeta_5_4_3_C_0.tiff
#   After removing extension and splitting by "_":
#     Well  = 4th element from the end  -> parts[-4]
#     Image = 3rd element from the end  -> parts[-3]
# ============================================================
from __future__ import annotations
import os
import pandas as pd
import numpy as np


# --- helper function to avoid overwriting files ---
def _unique_output_path(path: str) -> str:
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    counter = 1
    new_path = f"{base} ({counter}){ext}"
    while os.path.exists(new_path):
        counter += 1
        new_path = f"{base} ({counter}){ext}"
    return new_path


def _well_sort_key(x: str):
    try:
        return (0, int(x))
    except ValueError:
        return (1, x)


def _normalize_filename_key(name: str) -> str:
    """
    Normalize filenames from both Treatment sheet and data sheets so they match.
    - strips whitespace
    - uses basename only (drops folder)
    - removes one extension (".tiff", ".csv", etc.)
    """
    name = str(name).strip()
    name = os.path.basename(name)
    name = os.path.splitext(name)[0]
    return name


def load_treatment_map_from_sheet(
    input_excel_path: str,
    *,
    treatment_sheet_name: str = "Treatment",
    filename_col: int = 0,
    treatment_col: int = 1,
    biorep_col: int = 2,
    on_duplicate: str = "first",  # "first" | "last" | "error"
) -> dict[str, tuple[str, int | None]]:
    """
    SINGLE mapping (no row expansion):
      normalized_filename_key -> (treatment, biological_rep)

    If duplicates exist in the Treatment sheet for the same normalized key, we:
      - keep first (default), or last, or error.
    """
    df_map = pd.read_excel(input_excel_path, sheet_name=treatment_sheet_name)

    if df_map.shape[1] <= max(filename_col, treatment_col, biorep_col):
        raise ValueError(
            f"Treatment sheet '{treatment_sheet_name}' does not have enough columns. "
            f"Need at least {max(filename_col, treatment_col, biorep_col) + 1} column(s)."
        )

    mapping: dict[str, tuple[str, int | None]] = {}
    skipped = 0
    duplicates = 0

    for i in range(len(df_map)):
        fname = df_map.iloc[i, filename_col]
        trt = df_map.iloc[i, treatment_col]
        bio = df_map.iloc[i, biorep_col]

        if pd.isna(fname) or pd.isna(trt):
            skipped += 1
            continue

        key = _normalize_filename_key(fname)
        treatment = str(trt).strip()

        bio_rep: int | None = None
        if not pd.isna(bio):
            try:
                bio_rep = int(bio)
            except (ValueError, TypeError):
                bio_rep = None

        if key in mapping:
            duplicates += 1
            if on_duplicate == "first":
                continue
            if on_duplicate == "last":
                mapping[key] = (treatment, bio_rep)
                continue
            if on_duplicate == "error":
                raise ValueError(f"Duplicate filename key in Treatment sheet: '{key}'")
        else:
            mapping[key] = (treatment, bio_rep)

    if not mapping:
        raise ValueError(f"No usable filename->(treatment,biorep) rows found in sheet '{treatment_sheet_name}'.")

    if skipped:
        print(f"⚠️  Treatment map: skipped {skipped} row(s) with missing filename or treatment.")
    if duplicates:
        print(
            f"⚠️  Treatment map: {duplicates} duplicate filename key(s) found in Treatment sheet. "
            f"Kept '{on_duplicate}' occurrence to avoid inflating output rows."
        )

    print(f"✓ Loaded treatment map from sheet '{treatment_sheet_name}': {len(mapping)} unique filename key(s)")
    return mapping


def split_excel_sheet(
    df: pd.DataFrame,
    sheet_name: str,
    input_excel_path: str,
    filename_col: int = 0,
    value_col: int = 1,
    overwrite: bool = False,
    biological_rep=None,  # fallback only
    start_well: int = 1,
    *,
    treatment_map: dict[str, tuple[str, int | None]] | None = None,
):
    """
    Process a single sheet and write a formatted CSV.

    Filename parsing rule (UPDATED):
      Well  = 4th element from the end  -> parts[-4]
      Image = 3rd element from the end  -> parts[-3]

    Example:
      TgDIV18_032124_WT11_Abeta_5_4_3_C_0.tiff
        key   = TgDIV18_032124_WT11_Abeta_5_4_3_C_0
        parts = [..., '5','4','3','C','0']
        Well  = '3'  (parts[-4])
        Image = 'C'  (parts[-3])
    """
    if df.shape[1] < 2:
        print(f"  ⚠️  Warning: Sheet has only {df.shape[1]} column(s). Need at least 2 (filename + results).")
        return pd.DataFrame(), None, start_well

    filename_col_name = df.columns[filename_col] if filename_col < len(df.columns) else f"Column {filename_col}"
    value_col_name = df.columns[value_col] if value_col < len(df.columns) else f"Column {value_col}"

    print(f"  Column {filename_col} (Filename): '{filename_col_name}'")
    print(f"  Column {value_col} (Results):   '{value_col_name}'")

    out_rows = []
    skipped_rows = 0
    missing_trt_rows = 0
    missing_bio_rows = 0
    parse_warn_rows = 0

    for idx in range(len(df)):
        name = df.iloc[idx, filename_col]
        if pd.isna(name):
            skipped_rows += 1
            continue
        name = str(name).strip()

        key = _normalize_filename_key(name)
        parts = key.split("_")

        # --- CHANGED: Well is 4th from end and Image is 3rd from end ---
        if len(parts) >= 4:
            well_token = parts[-4]   # 4th from end
            image_token = parts[-3]  # 3rd from end
        else:
            parse_warn_rows += 1
            well_token = None
            image_token = None

        try:
            well_num = int(well_token) if well_token is not None else None
        except (ValueError, TypeError):
            well_num = well_token

        try:
            image_num = int(image_token) if image_token is not None else None
        except (ValueError, TypeError):
            image_num = image_token

        result_val = df.iloc[idx, value_col]
        if pd.isna(result_val):
            print(f"  ⚠️  Row {idx+1}: Missing result value for {name}")
            skipped_rows += 1
            continue

        treatment = "Unknown"
        bio_rep_row = biological_rep

        if treatment_map is not None:
            hit = treatment_map.get(key, None)
            if hit is None:
                missing_trt_rows += 1
            else:
                treatment, bio_from_map = hit
                if bio_from_map is not None:
                    bio_rep_row = bio_from_map

        if bio_rep_row is None:
            missing_bio_rows += 1

        out_rows.append(
            {
                "Filename": name,
                "Treatment": treatment,
                "Biological_Rep": bio_rep_row,
                "Well": well_num,
                "Image": image_num,
                "Results": result_val,
                "Original_Order": idx,
            }
        )

    if skipped_rows > 0:
        print(f"  ⚠️  Skipped {skipped_rows} row(s) due to missing filename or result value")
    if missing_trt_rows > 0 and treatment_map is not None:
        print(f"  ⚠️  {missing_trt_rows} row(s) had no filename match in Treatment sheet; Treatment set to 'Unknown'")
    if missing_bio_rows > 0:
        print(f"  ⚠️  {missing_bio_rows} row(s) have no Biological_Rep (neither from map nor fallback)")
    if parse_warn_rows > 0:
        print(f"  ⚠️  {parse_warn_rows} row(s) had too few '_' parts to parse Well/Image by position; set to None")

    out_df = pd.DataFrame(out_rows)
    if len(out_df) == 0:
        print("  ⚠️  No valid rows to export")
        return pd.DataFrame(), None, start_well

    out_df = out_df.sort_values(["Original_Order"], na_position="last").reset_index(drop=True)
    out_df = out_df[["Filename", "Treatment", "Biological_Rep", "Well", "Image", "Results"]]

    in_dir = os.path.dirname(os.path.abspath(input_excel_path))
    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]
    safe_sheet_name = str(sheet_name).replace("/", "_").replace("\\", "_")
    out_path = os.path.join(in_dir, f"{in_stem}_{safe_sheet_name}_RMixedFormated.csv")
    if not overwrite:
        out_path = _unique_output_path(out_path)

    out_df.to_csv(out_path, index=False)
    print(f"\n✓ Saved: {out_path}")
    print(f"  Sheet: {sheet_name}")
    print(f"  Total rows: {len(out_df)}")
    return out_df, out_path, start_well


def split_excel_all_sheets(
    input_excel_path: str,
    filename_col: int = 0,
    value_col: int = 1,
    overwrite: bool = False,
    biological_reps=None,
    auto_number_bio_reps: bool = True,
    *,
    treatment_sheet_name: str = "Treatment",
):
    excel_file = pd.ExcelFile(input_excel_path)
    sheet_names = excel_file.sheet_names

    treatment_map = None
    if treatment_sheet_name in sheet_names:
        treatment_map = load_treatment_map_from_sheet(
            input_excel_path,
            treatment_sheet_name=treatment_sheet_name,
            filename_col=filename_col,
            treatment_col=1,
            biorep_col=2,
            on_duplicate="first",
        )
    else:
        print(f"⚠️  No '{treatment_sheet_name}' sheet found; Treatment will be 'Unknown'.")

    data_sheet_names = [s for s in sheet_names if s != treatment_sheet_name]

    results = {}
    for sheet_name in data_sheet_names:
        print(f"\n{'─'*60}")
        print(f"Processing sheet: {sheet_name}")
        print(f"{'─'*60}")

        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

        # fallback only (normally BioRep comes from Treatment sheet col C)
        if isinstance(biological_reps, dict):
            bio_rep = biological_reps.get(sheet_name, None)
        elif isinstance(biological_reps, int):
            bio_rep = biological_reps
        else:
            bio_rep = None

        out_df, out_path, _ = split_excel_sheet(
            df=df,
            sheet_name=sheet_name,
            input_excel_path=input_excel_path,
            filename_col=filename_col,
            value_col=value_col,
            overwrite=overwrite,
            biological_rep=bio_rep,
            treatment_map=treatment_map,
        )

        if out_path is not None:
            results[sheet_name] = (out_df, out_path)

    return results


# ================================================================================
# CALL THE SOURCE EXCEL FILE
# ================================================================================
EXCEL_FILE_PATH = '/Users/allisonpickle/Desktop/By LPS Concentration/Treatment per LPS Concentration.xlsx'

if not os.path.exists(EXCEL_FILE_PATH):
    print(f"ERROR: File not found at {EXCEL_FILE_PATH}")
    print("Please check the file path.")
else:
    results = split_excel_all_sheets(
        EXCEL_FILE_PATH,
        filename_col=0,
        value_col=1,
        biological_reps=None,
        treatment_sheet_name="Treatment",
    )

⚠️  No 'Treatment' sheet found; Treatment will be 'Unknown'.

────────────────────────────────────────────────────────────
Processing sheet: Ctrl
────────────────────────────────────────────────────────────
  Column 0 (Filename): 'File'
  Column 1 (Results):   'Base Name'
  ⚠️  254 row(s) have no Biological_Rep (neither from map nor fallback)

✓ Saved: /Users/allisonpickle/Desktop/By LPS Concentration/Treatment per LPS Concentration_Ctrl_RMixedFormated.csv
  Sheet: Ctrl
  Total rows: 254

────────────────────────────────────────────────────────────
Processing sheet: 5ng
────────────────────────────────────────────────────────────
  Column 0 (Filename): 'File'
  Column 1 (Results):   'Base Name'
  ⚠️  253 row(s) have no Biological_Rep (neither from map nor fallback)

✓ Saved: /Users/allisonpickle/Desktop/By LPS Concentration/Treatment per LPS Concentration_5ng_RMixedFormated.csv
  Sheet: 5ng
  Total rows: 253

────────────────────────────────────────────────────────────
Processing sheet

260507 Sholl Analysis Skeletonization Comparison

In [3]:
import os
import pandas as pd
import numpy as np

# --- helper function to avoid overwriting files ---
def _unique_output_path(path):
    """Generates a unique file path by appending a number if the file exists."""
    if not os.path.exists(path):
        return path
    base, ext = os.path.splitext(path)
    counter = 1
    new_path = f"{base} ({counter}){ext}"
    while os.path.exists(new_path):
        counter += 1
        new_path = f"{base} ({counter}){ext}"
    return new_path


def _sanitize_filename_part(s: str) -> str:
    """Make a string safe-ish for filenames."""
    s = str(s).strip()
    for ch in ["/", "\\", ":", "*", "?", "\"", "<", ">", "|"]:
        s = s.replace(ch, "_")
    s = " ".join(s.split())
    return s


def split_excel_sheet_auc_wide_to_mixed_csv(
    df: pd.DataFrame,
    sheet_name: str,
    input_excel_path: str,
    key_col: str = "Key",
    auc_cols=("AUC_Skeletonize", "AUC_Thin", "AUC_MedialAxis"),
    overwrite: bool = False,
    biological_rep=None,
    output_dir=None,
    include_sheet_name_in_output: bool = True,
):
    """
    Converts a sheet with columns:
      Key | AUC_Skeletonize | AUC_Thin | AUC_MedialAxis

    into ONE long CSV suitable for mixed-effects modeling:
      Key, Treatment, Well, Image, Biological_Rep, Method, Results

    - Method is one of: Skeletonize / Thin / MedialAxis
    - Results is the corresponding AUC value
    """
    if key_col not in df.columns:
        raise ValueError(f"Sheet '{sheet_name}': missing required column '{key_col}'. Found: {list(df.columns)}")

    missing_auc = [c for c in auc_cols if c not in df.columns]
    if missing_auc:
        raise ValueError(f"Sheet '{sheet_name}': missing AUC columns {missing_auc}. Found: {list(df.columns)}")

    # ---- choose output directory ----
    if output_dir is None:
        out_root = os.path.dirname(os.path.abspath(input_excel_path))
    else:
        out_root = os.path.abspath(output_dir)
        os.makedirs(out_root, exist_ok=True)

    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]
    safe_sheet_name = _sanitize_filename_part(sheet_name)

    out_rows = []
    skipped_key = 0
    skipped_all_nan = 0

    for idx in range(len(df)):
        key = df.loc[df.index[idx], key_col]
        if pd.isna(key):
            skipped_key += 1
            continue

        key = str(key).strip()
        base = os.path.splitext(os.path.basename(key))[0]
        parts = base.split("_")

        # Expected patterns like: Tri_Ctrl_01_1 or Tri_5ugLPS_01_1
        treatment = f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else (parts[0] if parts else "Unknown")
        original_well_id = parts[2] if len(parts) > 2 else "1"
        image_id = parts[3] if len(parts) > 3 else "1"

        # keep your existing mapping logic (as-is)
        try:
            original_well_num = int(original_well_id)
            well_num = ((original_well_num - 1) // 4) % 3 + 1
        except (ValueError, TypeError):
            well_num = original_well_id

        try:
            image_num = int(image_id)
        except (ValueError, TypeError):
            image_num = image_id

        any_written = False
        for c in auc_cols:
            val = pd.to_numeric(df.loc[df.index[idx], c], errors="coerce")
            if pd.isna(val):
                continue

            out_rows.append(
                {
                    "Key": key,
                    "Treatment": treatment,
                    "Biological_Rep": biological_rep,
                    "Well": well_num,
                    "Image": image_num,
                    "Method": str(c).replace("AUC_", ""),
                    "Results": float(val),
                }
            )
            any_written = True

        if not any_written:
            skipped_all_nan += 1

    out_df = pd.DataFrame(out_rows)
    if out_df.empty:
        print(f"  No valid rows to export for sheet '{sheet_name}'.")
        return None, None

    # stable ordering
    out_df["__order"] = range(len(out_df))
    out_df = out_df.drop(columns=["__order"])

    # output path
    if include_sheet_name_in_output:
        out_path = os.path.join(out_root, f"{in_stem}_{safe_sheet_name}_AUC_Long.csv")
    else:
        out_path = os.path.join(out_root, f"{in_stem}_AUC_Long.csv")

    if not overwrite:
        out_path = _unique_output_path(out_path)

    out_df.to_csv(out_path, index=False)

    print(f"\n✓ Saved: {out_path}")
    print(f"  Sheet: {sheet_name}")
    print(f"  Total rows: {len(out_df)} (Key×Method long format)")
    if skipped_key:
        print(f"  Skipped {skipped_key} row(s) due to missing Key")
    if skipped_all_nan:
        print(f"  Skipped {skipped_all_nan} row(s) where all AUC values were missing")

    return out_df, out_path


def convert_auc_excel_all_sheets_to_mixed_csv(
    input_excel_path: str,
    key_col: str = "Key",
    auc_cols=("AUC_Skeletonize", "AUC_Thin", "AUC_MedialAxis"),
    overwrite: bool = False,
    biological_reps=None,              # int OR dict[sheet->int] OR None to prompt
    output_dir=None,
    include_sheet_name_in_output: bool = True,
):
    excel_file = pd.ExcelFile(input_excel_path)
    sheet_names = excel_file.sheet_names

    print(f"\n{'='*60}")
    print("Excel AUC (wide) → Mixed Effects CSV (long)")
    print(f"{'='*60}")
    print(f"Input file: {os.path.basename(input_excel_path)}")
    print(f"Found {len(sheet_names)} sheet(s):")
    for i, name in enumerate(sheet_names, 1):
        print(f"  {i}. {name}")
    print()
    print("Expected columns per sheet:")
    print(f"  Key column: {key_col}")
    print(f"  AUC columns: {list(auc_cols)}")
    if output_dir is not None:
        print(f"Output dir: {os.path.abspath(output_dir)}")

    all_results = {}

    for sheet_name in sheet_names:
        print(f"\n{'─'*60}")
        print(f"Processing sheet: {sheet_name}")
        print(f"{'─'*60}")

        df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

        if isinstance(biological_reps, dict):
            bio_rep = biological_reps.get(sheet_name, None)
        elif isinstance(biological_reps, int):
            bio_rep = biological_reps
        else:
            while True:
                try:
                    bio_rep = int(input(f"Enter Biological Replicate number for sheet '{sheet_name}': "))
                    break
                except ValueError:
                    print("Please enter a valid integer number.")

        out_df, out_path = split_excel_sheet_auc_wide_to_mixed_csv(
            df=df,
            sheet_name=sheet_name,
            input_excel_path=input_excel_path,
            key_col=key_col,
            auc_cols=auc_cols,
            overwrite=overwrite,
            biological_rep=bio_rep,
            output_dir=output_dir,
            include_sheet_name_in_output=include_sheet_name_in_output,
        )

        if out_df is not None:
            all_results[sheet_name] = (out_df, out_path)

    print(f"\n{'='*60}")
    print(f"SUMMARY: Processed {len(all_results)} sheet(s)")
    print(f"{'='*60}")
    for sheet_name, (df_out, path) in all_results.items():
        print(f"  {sheet_name}: {len(df_out)} rows → {os.path.basename(path)}")
    print(f"{'='*60}\n")

    return all_results



# ================================================================================
# CALL THE SOURCE EXCEL FILE  (UPDATE THESE PATHS)
# ================================================================================
EXCEL_FILE_PATH = '/Users/allisonpickle/Desktop/Sampling Density/Compiled Step Sizes.xlsx'
if not os.path.exists(EXCEL_FILE_PATH):
    print(f"ERROR: File not found at {EXCEL_FILE_PATH}")
else:
    results = convert_auc_excel_all_sheets_to_mixed_csv(
        EXCEL_FILE_PATH,
        key_col="Key",
        auc_cols=("AUC_Skeletonize", "AUC_Thin", "AUC_MedialAxis"),
        overwrite=False,
        biological_reps=1,                   # or a dict per sheet
        output_dir='/Users/allisonpickle/Desktop/Sampling Density',
        include_sheet_name_in_output=True,
    )


Excel AUC (wide) → Mixed Effects CSV (long)
Input file: Compiled Step Sizes.xlsx
Found 1 sheet(s):
  1. Sheet1

Expected columns per sheet:
  Key column: Key
  AUC columns: ['AUC_Skeletonize', 'AUC_Thin', 'AUC_MedialAxis']
Output dir: /Users/allisonpickle/Desktop/Sampling Density

────────────────────────────────────────────────────────────
Processing sheet: Sheet1
────────────────────────────────────────────────────────────


ValueError: Sheet 'Sheet1': missing AUC columns ['AUC_Skeletonize', 'AUC_Thin', 'AUC_MedialAxis']. Found: ['Key', 'Step Size', 'AUC']

For sholl analysis of starting radius

In [ ]:
# ...existing code...
import os
import numpy as np
import pandas as pd

def convert_stepsize_excel_to_mixed_csv_single_sheet(
    input_excel_path: str,
    sheet_name: str = 0,                 # 0 = first sheet
    key_col: str = "Key",
    stepsize_col_positions=(1, 2),       # positions of the TWO step-size columns (0-based)
    stepsize_labels=("S2.5", "S5"),      # labels you want in output
    overwrite: bool = False,
    biological_rep=None,
    output_dir=None,
    include_sheet_name_in_output: bool = False,
):
    """
    Reads ONE sheet and converts ONLY the two step-size columns (by POSITION) into long format.

    Input expected (example):
      Col A: Key
      Col B: step size = 2.5
      Col C: step size = 5

    Output columns:
      Key, Treatment, Well, Image, Biological_Rep, StepSize, Results
    """
    df = pd.read_excel(input_excel_path, sheet_name=sheet_name)

    if key_col not in df.columns:
        raise ValueError(f"Missing '{key_col}'. Found columns: {list(df.columns)}")

    if len(stepsize_col_positions) != len(stepsize_labels):
        raise ValueError("stepsize_col_positions and stepsize_labels must be the same length.")

    # Resolve by POSITIONS only (robust even when headers are numbers like 2.5 and 5)
    resolved = []
    for pos in stepsize_col_positions:
        if not isinstance(pos, (int, np.integer)):
            raise ValueError(f"stepsize_col_positions must be integers; got {pos} ({type(pos)})")
        if pos < 0 or pos >= len(df.columns):
            raise ValueError(f"StepSize column index {pos} out of range for df with {len(df.columns)} columns.")
        resolved.append(df.columns[int(pos)])

    # output dir
    if output_dir is None:
        out_root = os.path.dirname(os.path.abspath(input_excel_path))
    else:
        out_root = os.path.abspath(output_dir)
        os.makedirs(out_root, exist_ok=True)

    in_stem = os.path.splitext(os.path.basename(input_excel_path))[0]

    out_rows = []
    skipped_key = 0
    skipped_all_nan = 0

    for i in range(len(df)):
        key = df.loc[df.index[i], key_col]
        if pd.isna(key):
            skipped_key += 1
            continue

        key = str(key).strip()
        base = os.path.splitext(os.path.basename(key))[0]
        parts = base.split("_")

        # keep same parsing logic as your radius version
        treatment = f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else (parts[0] if parts else "Unknown")
        original_well_id = parts[2] if len(parts) > 2 else "1"
        image_id = parts[3] if len(parts) > 3 else "1"

        try:
            original_well_num = int(original_well_id)
            well_num = ((original_well_num - 1) // 4) % 3 + 1
        except (ValueError, TypeError):
            well_num = original_well_id

        try:
            image_num = int(image_id)
        except (ValueError, TypeError):
            image_num = image_id

        any_written = False
        for col_name, ss_label in zip(resolved, stepsize_labels):
            val = pd.to_numeric(df.loc[df.index[i], col_name], errors="coerce")
            if pd.isna(val):
                continue

            out_rows.append(
                {
                    "Key": key,
                    "Treatment": treatment,
                    "Biological_Rep": biological_rep,
                    "Well": well_num,
                    "Image": image_num,
                    "StepSize": ss_label,
                    "Results": float(val),
                }
            )
            any_written = True

        if not any_written:
            skipped_all_nan += 1

    out_df = pd.DataFrame(out_rows)
    if out_df.empty:
        raise ValueError("No valid rows to export (all missing / filtered).")

    out_path = os.path.join(out_root, f"{in_stem}_StepSize_Long.csv")
    if include_sheet_name_in_output:
        safe_sheet = _sanitize_filename_part(sheet_name)
        out_path = os.path.join(out_root, f"{in_stem}_{safe_sheet}_StepSize_Long.csv")

    if not overwrite:
        out_path = _unique_output_path(out_path)

    out_df.to_csv(out_path, index=False)

    print(f"\n✓ Saved: {out_path}")
    print(f"  Sheet read: {sheet_name}")
    print(f"  Columns used (by position): {list(stepsize_col_positions)} -> headers {resolved}")
    print(f"  Labels: {list(stepsize_labels)}")
    print(f"  Total rows: {len(out_df)} (Key×StepSize long format)")
    if skipped_key:
        print(f"  Skipped {skipped_key} row(s) due to missing Key")
    if skipped_all_nan:
        print(f"  Skipped {skipped_all_nan} row(s) where both step-size values were missing")

    return out_df, out_path

# ---- RUN ----
EXCEL_FILE_PATH = "/Users/allisonpickle/Desktop/Sholl Analysis by Starting Radius.xlsx"
OUTPUT_DIR = '/Users/allisonpickle/Desktop/Isolated Microglia'

out_df, out_path = convert_stepsize_excel_to_mixed_csv_single_sheet(
    input_excel_path=EXCEL_FILE_PATH,
    sheet_name=0,                    # first sheet
    key_col="Key",
    stepsize_col_positions=(1, 2),   # B and C
    stepsize_labels=("S2.5", "S5"),  # will appear in output as StepSize
    overwrite=False,
    biological_rep=1,
    output_dir=OUTPUT_DIR,
    include_sheet_name_in_output=False,
)

print("Saved:", out_path)
print(out_df.head())



✓ Saved: /Users/allisonpickle/Desktop/Sholl Analysis by step size/Sholl Analysis by Starting Radius_StepSize_Long.csv
  Sheet read: 0
  Columns used (by position): [1, 2] -> headers [0, 9.81]
  Labels: ['S2.5', 'S5']
  Total rows: 540 (Key×StepSize long format)
Saved: /Users/allisonpickle/Desktop/Sholl Analysis by step size/Sholl Analysis by Starting Radius_StepSize_Long.csv
               Key   Treatment  Biological_Rep  Well  Image StepSize   Results
0  Tri_5ugLPS_01_1  Tri_5ugLPS               1     1      1     S2.5  215.7400
1  Tri_5ugLPS_01_1  Tri_5ugLPS               1     1      1       S5  147.7350
2  Tri_5ugLPS_01_2  Tri_5ugLPS               1     1      2     S2.5  206.3600
3  Tri_5ugLPS_01_2  Tri_5ugLPS               1     1      2       S5  144.2175
4  Tri_5ugLPS_02_1  Tri_5ugLPS               1     1      1     S2.5  673.0150

✓ Saved: /Users/allisonpickle/Desktop/Sholl Analysis by starting radius/Sholl Analysis by Step Size_StepSize_Long (1).csv
  Sheet read: 0
  Column

In [6]:
def compile_barplots_by_metric(
    parent_folder,
    plot_type="LogFit",  # "LogFit" or "RawFit"
    plate_pattern="Plate",  # What to look for in folder names to identify plates
    output_folder=None,
    figsize_per_plot=(8, 6),
    dpi=300
):
    """
    Compile barplots from multiple plate subfolders into single figures grouped by metric.
    
    Folder structure expected:
        parent_folder/
            MM_Outputs_Plate 14_Aspect Ratio/
                Barplot_LogFit.png
                Barplot_RawFit.png
            MM_Outputs_Plate 15_Aspect Ratio/
                Barplot_LogFit.png
                Barplot_RawFit.png
            MM_Outputs_Plate 14_Area/
                Barplot_LogFit.png
                Barplot_RawFit.png
            ...
    
    This will create compiled figures like:
        Compiled_Aspect_Ratio_LogFit.png (contains Plate 14, 15, 16... Aspect Ratio plots)
        Compiled_Area_LogFit.png (contains Plate 14, 15, 16... Area plots)
    
    Parameters
    ----------
    parent_folder : str
        Path to folder containing all MM_Outputs subfolders
    plot_type : str
        "LogFit" or "RawFit" - which barplot type to compile
    plate_pattern : str
        String pattern to identify plate folders (default: "Plate")
    output_folder : str, optional
        Where to save compiled figures. If None, saves to parent_folder
    figsize_per_plot : tuple
        Size of each individual plot in inches (width, height)
    dpi : int
        Resolution for output images
        
    Returns
    -------
    dict
        Dictionary mapping metric names to output file paths
    """
    import os
    from pathlib import Path
    import re
    from PIL import Image
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    
    parent_folder = Path(parent_folder)
    
    if output_folder is None:
        output_folder = parent_folder / "Compiled_Barplots"
    else:
        output_folder = Path(output_folder)
    
    output_folder.mkdir(exist_ok=True, parents=True)
    
    # Expected image filename
    image_filename = f"Barplot_{plot_type}.png"
    
    # Find all subfolders containing the plate pattern
    all_folders = [f for f in parent_folder.iterdir() if f.is_dir() and plate_pattern in f.name]
    
    if not all_folders:
        print(f"No folders found containing '{plate_pattern}' in {parent_folder}")
        return {}
    
    print(f"\nFound {len(all_folders)} folders containing '{plate_pattern}'")
    
    # Parse folder names to extract plate number and metric
    # Expected format: "MM_Outputs_Plate 14_Aspect Ratio"
    folder_info = []
    
    for folder in all_folders:
        # Split by underscore
        parts = folder.name.split('_')
        
        # Find the plate number
        plate_num = None
        plate_idx = None
        for i, part in enumerate(parts):
            if plate_pattern in part:
                # Try to extract number from this part or next part
                match = re.search(r'(\d+)', part)
                if match:
                    plate_num = int(match.group(1))
                    plate_idx = i
                    break
                # Check next part if exists
                if i + 1 < len(parts):
                    match = re.search(r'(\d+)', parts[i + 1])
                    if match:
                        plate_num = int(match.group(1))
                        plate_idx = i
                        break
        
        if plate_num is None:
            print(f"Warning: Could not extract plate number from {folder.name}")
            continue
        
        # Metric is everything after the plate specification
        if plate_idx is not None and plate_idx + 2 < len(parts):
            metric = '_'.join(parts[plate_idx + 2:])
        else:
            # Fallback: use last part
            metric = parts[-1]
        
        # Check if the image exists
        image_path = folder / image_filename
        if not image_path.exists():
            print(f"Warning: {image_filename} not found in {folder.name}")
            continue
        
        folder_info.append({
            'folder': folder,
            'plate': plate_num,
            'metric': metric,
            'image_path': image_path
        })
    
    if not folder_info:
        print("No valid folders with images found")
        return {}
    
    # Group by metric
    from collections import defaultdict
    metric_groups = defaultdict(list)
    
    for info in folder_info:
        metric_groups[info['metric']].append(info)
    
    print(f"\nFound {len(metric_groups)} unique metrics:")
    for metric in metric_groups.keys():
        print(f"  - {metric}")
    
    # Sort each group by plate number
    for metric in metric_groups:
        metric_groups[metric].sort(key=lambda x: x['plate'])
    
    # Create compiled figures
    output_paths = {}
    
    for metric, plate_list in metric_groups.items():
        print(f"\n{'='*60}")
        print(f"Compiling {metric} ({len(plate_list)} plates)")
        print(f"{'='*60}")
        
        # Load all images
        images = []
        plate_labels = []
        
        for info in plate_list:
            try:
                img = Image.open(info['image_path'])
                images.append(img)
                plate_labels.append(f"Plate {info['plate']}")
                print(f"  ✓ Loaded: Plate {info['plate']} - {info['image_path'].name}")
            except Exception as e:
                print(f"  ✗ Error loading {info['image_path']}: {e}")
        
        if not images:
            print(f"  No images loaded for {metric}, skipping")
            continue
        
        # Determine grid layout
        n_images = len(images)
        n_cols = min(2, n_images)  # Max 2 columns
        n_rows = (n_images + n_cols - 1) // n_cols
        
        # Create figure
        fig_width = figsize_per_plot[0] * n_cols
        fig_height = figsize_per_plot[1] * n_rows
        
        fig = plt.figure(figsize=(fig_width, fig_height))
        gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, 
                              hspace=0.1, wspace=0.1)  # Changed from 0.3 and 0.2
        
        # Add each image
        for idx, (img, label) in enumerate(zip(images, plate_labels)):
            row = idx // n_cols
            col = idx % n_cols
            
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(label, fontsize=14, fontweight='bold', pad=10)
        
        # Overall title
        fig.suptitle(f"{metric} - {plot_type} Comparison", 
                    fontsize=16, fontweight='bold', y=0.98)

        
        # Save
        safe_metric = metric.replace(' ', '_').replace('/', '_')
        output_filename = f"Compiled_{safe_metric}_{plot_type}.png"
        output_path = output_folder / output_filename
        
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print(f"\n✓ Saved: {output_path}")
        print(f"  Grid: {n_rows} rows × {n_cols} cols")
        print(f"  Size: {fig_width:.1f} × {fig_height:.1f} inches")
        
        output_paths[metric] = str(output_path)
    
    print(f"\n{'='*60}")
    print(f"SUMMARY: Created {len(output_paths)} compiled figures")
    print(f"{'='*60}")
    print(f"Output folder: {output_folder}")
    for metric, path in output_paths.items():
        print(f"  - {metric}: {Path(path).name}")
    print(f"{'='*60}\n")
    
    return output_paths


# ================================================================================
# USAGE EXAMPLES
# ================================================================================

# Example 1: Compile LogFit barplots
output_paths = compile_barplots_by_metric(
    parent_folder='/Users/allisonpickle/Desktop/MM_Outputs',
    plot_type='LogFit',
    plate_pattern='Plate',
    figsize_per_plot=(8, 6),
    dpi=300
)

# Example 2: Compile RawFit barplots
output_paths = compile_barplots_by_metric(
    parent_folder='/Users/allisonpickle/Desktop/MM_Outputs',
    plot_type='RawFit',
    plate_pattern='Plate',
    figsize_per_plot=(8, 6),
    dpi=300
)

# Example 3: Custom output location
# output_paths = compile_barplots_by_metric(
#     parent_folder='/Users/allisonpickle/Desktop/MM_Outputs',
#     plot_type='LogFit',
#     output_folder='/Users/allisonpickle/Desktop/Compiled_Plots',
#     figsize_per_plot=(10, 7),
#     dpi=150
# )


Found 56 folders containing 'Plate'

Found 7 unique metrics:
  - Circularity
  - Aspect Ratio
  - Area (µm²)
  - Perimeter (µm)
  - Roughness
  - Convex Hull Perimeter (µm)
  - Convex Hull Area (µm²)

Compiling Circularity (8 plates)
  ✓ Loaded: Plate 14 - Barplot_LogFit.png
  ✓ Loaded: Plate 15 - Barplot_LogFit.png
  ✓ Loaded: Plate 16 - Barplot_LogFit.png
  ✓ Loaded: Plate 17 - Barplot_LogFit.png
  ✓ Loaded: Plate 18 - Barplot_LogFit.png
  ✓ Loaded: Plate 19 - Barplot_LogFit.png
  ✓ Loaded: Plate 20 - Barplot_LogFit.png
  ✓ Loaded: Plate 21 - Barplot_LogFit.png

✓ Saved: /Users/allisonpickle/Desktop/MM_Outputs/Compiled_Barplots/Compiled_Circularity_LogFit.png
  Grid: 4 rows × 2 cols
  Size: 16.0 × 24.0 inches

Compiling Aspect Ratio (8 plates)
  ✓ Loaded: Plate 14 - Barplot_LogFit.png
  ✓ Loaded: Plate 15 - Barplot_LogFit.png
  ✓ Loaded: Plate 16 - Barplot_LogFit.png
  ✓ Loaded: Plate 17 - Barplot_LogFit.png
  ✓ Loaded: Plate 18 - Barplot_LogFit.png
  ✓ Loaded: Plate 19 - Barplot_Lo